In [ ]:
# Libraries
import os
import gc
import random
import numpy as np
import scipy.io as scio
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Conv2D, MaxPooling2D,
    Bidirectional, LSTM, GRU, TimeDistributed, Flatten)
from tensorflow.keras.models import Model
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

In [ ]:
# Dataset Configuration
data_bvp   = 'BVPwithout2/'              # Path to dataset root folder, Dataset 1 only
ALL_MOTION = list(range(1, 13))          # Motion labels 1–12
N_MOTION   = len(ALL_MOTION)             # = 12
T_MAX      = 0                           # Will be computed from data

In [ ]:
# Activity Label Mapping per Session 
# Each session recorded different subsets of 12 activities.
# This maps original per-session labels → unified label names.
activity_mapping = {
    '20181109-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181115-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181117-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181118-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181121-VS': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'},
    '20181127-VS': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'},
    '20181128-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Horizontal)', 5: 'Draw-Zigzag(Horizontal)', 6: 'Draw-N(Horizontal)'},
    '20181130-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)', 7: 'Draw-N(Horizontal)', 8: 'Draw-Triangle(Horizontal)', 9: 'Draw-Rectangle(Horizontal)'},
    '20181204-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)', 7: 'Draw-N(Horizontal)', 8: 'Draw-Triangle(Horizontal)', 9: 'Draw-Rectangle(Horizontal)'},
    
    # SPECIAL SESSION: User-specific nested mapping
    '20181205-VS': {
        'user2': {1: 'Draw-O(Horizontal)', 2: 'Draw-Zigzag(Horizontal)', 3: 'Draw-N(Horizontal)', 4: 'Draw-Triangle(Horizontal)', 5: 'Draw-Rectangle(Horizontal)'},
        'user3': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'}},
    
    '20181208-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide'},
    '20181209-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)'},
    '20181211-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)'}
}


# Unified Label Set (12 classes)
standardized_labels = {
    'Push&Pull': 1,
    'Sweep': 2,
    'Clap': 3,
    'Slide': 4,
    'Draw-O(Horizontal)': 5,
    'Draw-Zigzag(Horizontal)': 6,
    'Draw-Zigzag(Vertical)': 7,
    'Draw-N(Vertical)': 8,
    'Draw-O(Vertical)': 9,
    'Draw-N(Horizontal)': 10,
    'Draw-Triangle(Horizontal)': 11,
    'Draw-Rectangle(Horizontal)': 12
}

# Session → Date Mapping (for chronological sorting) 
session_dates = {
    '20181109-VS': '20181109',
    '20181115-VS': '20181115',
    '20181117-VS': '20181117',
    '20181118-VS': '20181118',
    '20181121-VS': '20181121',
    '20181127-VS': '20181127',
    '20181128-VS': '20181128',
    '20181130-VS': '20181130',
    '20181204-VS': '20181204',
    '20181205-VS': '20181205',
    '20181208-VS': '20181208',
    '20181209-VS': '20181209',
    '20181211-VS': '20181211'
}

# Room → User Mapping 
room_user_mapping = {
    'room1': ['user1', 'user2', 'user3', 'user5', 'user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
    'room2': ['user1', 'user2', 'user3', 'user4', 'user5', 'user6'],
    'room3': ['user3', 'user7', 'user8', 'user9']
}

# Session → Room Mapping 
subfolder_room_mapping = {
    '20181109-VS': 'room1',
    '20181115-VS': 'room1',
    '20181117-VS': 'room2',
    '20181118-VS': 'room2',
    '20181121-VS': 'room1',
    '20181127-VS': 'room2',
    '20181128-VS': 'room2',
    '20181130-VS': 'room1',
    '20181204-VS': 'room2',
    '20181205-VS': 'room2',
    '20181208-VS': 'room2',
    '20181209-VS': 'room2',
    '20181211-VS': 'room3'
}

# Check samples are coming from which room
def get_user_rooms(user_id, room_user_mapping):
    rooms = []
    for room, users in room_user_mapping.items():
        if user_id in users:
            rooms.append(room)
    return rooms

# Function to standardize activity labels
def standardize_activity_labels(date, activity_label, user_id):

    if date not in activity_mapping:
        return None
    
    session_mapping = activity_mapping[date]
    # Check if this session has user-specific mappings
    first_key = next(iter(session_mapping.keys()))
    
    if isinstance(first_key, str):  # User-specific mapping
        # This is a special session like 20181205-VS
        if user_id not in session_mapping:
            return None
        
        user_mapping = session_mapping[user_id]
        if activity_label not in user_mapping:
            return None
        
        original_label = user_mapping[activity_label]
    else:  # Standard mapping
        if activity_label not in session_mapping:
            return None
        original_label = session_mapping[activity_label]
    
    if original_label in standardized_labels:
        return standardized_labels[original_label]
    
    return None


def determine_T_MAX(directories, motion_sel):
    """Scan all .mat files and return the maximum time dimension."""
    T_MAX = 0
    for directory in directories:
        for data_root, data_dirs, data_files in os.walk(directory):
            for data_file_name in data_files:
                file_path = os.path.join(data_root, data_file_name)
                try:
                    data_1 = scio.loadmat(file_path)['velocity_spectrum_ro']
                    label_1 = int(data_file_name.split('-')[1])
                    if label_1 not in motion_sel:
                        continue
                    T_MAX = max(T_MAX, data_1.shape[2])
                except Exception:
                    continue
    return T_MAX

def normalize_data(data_1):
    """Min-max normalize a 3-D sample array along the time axis."""
    data_1_max = np.concatenate((data_1.max(axis=0), data_1.max(axis=1)), axis=0).max(axis=0)
    data_1_min = np.concatenate((data_1.min(axis=0), data_1.min(axis=1)), axis=0).min(axis=0)
    if (len(np.where((data_1_max - data_1_min) == 0)[0]) > 0):
        return data_1
    data_1_max_rep = np.tile(data_1_max, (data_1.shape[0], data_1.shape[1], 1))
    data_1_min_rep = np.tile(data_1_min, (data_1.shape[0], data_1.shape[1], 1))
    data_1_norm = (data_1 - data_1_min_rep) / (data_1_max_rep - data_1_min_rep)
    return data_1_norm

def zero_padding(data, T_MAX):
    """Left-pad each sample along the time axis to length T_MAX."""
    data_pad = []
    for i in range(len(data)):
        t = np.array(data[i]).shape[2]
        data_pad.append(np.pad(data[i], ((0, 0), (0, 0), (T_MAX - t, 0)), 'constant', constant_values=0).tolist())
    return np.array(data_pad)

def onehot_encoding(label, num_class):
    """Convert integer labels (1-indexed) to one-hot vectors."""
    label = np.array(label).astype('int32')
    label = np.squeeze(label)
    _label = np.eye(num_class)[label - 1]  # from label to onehot
    return _label

def load_4D_data(directories, motion_sel, T_MAX, second_subfolder, user_id):
    """
    Load and preprocess all .mat files for one user-session.
    """
    data = []
    label = []
    skipped_count = 0
    
    for directory in directories:
        for data_root, data_dirs, data_files in os.walk(directory):
            for data_file_name in data_files:
                file_path = os.path.join(data_root, data_file_name)
                try:
                    data_1 = scio.loadmat(file_path)['velocity_spectrum_ro']
                    label_1 = int(data_file_name.split('-')[1])
                    
                    # Standardize label (now with user_id)
                    standardized_label_1 = standardize_activity_labels(
                        second_subfolder, 
                        label_1, 
                        user_id
                    )
                    
                    if standardized_label_1 is None:
                        skipped_count += 1
                        continue
                    
                    if standardized_label_1 not in motion_sel:
                        continue

                    data_normed_1 = normalize_data(data_1)
                    
                    # Save to list
                    data.append(data_normed_1.tolist())
                    label.append(standardized_label_1)
                    
                except Exception as e:
                    continue
    
    if skipped_count > 0:
        print(f"  Skipped {skipped_count} samples (unmapped labels)")
    
    if len(data) == 0:
        # Return empty arrays with correct shape
        return np.empty((0, 38, 20, 20, 1)), np.empty((0, 12))
    
    data = zero_padding(data, T_MAX)
    data = np.swapaxes(np.swapaxes(data, 1, 3), 2, 3)
    data = np.expand_dims(data, axis=-1)

    # Convert labels to ndarray
    label = np.array(label)
    label = onehot_encoding(label, len(standardized_labels))    
    return data, label




def load_all_user_data(base_directory, room_user_mapping, subfolder_room_mapping):
    """
    Walk the dataset directory, load every user-session, sort chronologically
    """    
    fl_clients = {}
    data_index = []

    # Step 1: Collect all user paths and compute global T_MAX 
    print("Step 1: Determining T_MAX...")
    all_directories = []
    for root, dirs, files in os.walk(base_directory):
        for subfolder in dirs:
            subfolder_path = os.path.join(root, subfolder)
            if os.path.isdir(subfolder_path):
                for second_subfolder in os.listdir(subfolder_path):
                    second_subfolder_path = os.path.join(subfolder_path, second_subfolder)
                    if os.path.isdir(second_subfolder_path):
                        for user_dir in os.listdir(second_subfolder_path):
                            if 'user' in user_dir:
                                user_path = os.path.join(second_subfolder_path, user_dir)
                                all_directories.append(user_path)
    
    T_MAX = determine_T_MAX(all_directories, ALL_MOTION)
    print(f"Global T_MAX = {T_MAX}\n")
    
    print("Loading data with session tracking...\n")
    # Temporary storage: {user_id: {room: [session_data_list]}}
    temp_storage = {}
    
    for root, dirs, files in os.walk(base_directory):
        for subfolder in dirs:
            subfolder_path = os.path.join(root, subfolder)
            if os.path.isdir(subfolder_path):
                for second_subfolder in sorted(os.listdir(subfolder_path)):
                    second_subfolder_path = os.path.join(subfolder_path, second_subfolder)
                    if os.path.isdir(second_subfolder_path):
                        
                        # Check if this is a valid session
                        if second_subfolder not in subfolder_room_mapping:
                            continue
                        
                        room = subfolder_room_mapping[second_subfolder]
                        session_date = session_dates.get(second_subfolder, second_subfolder[:8])
                        
                        for user_dir in sorted(os.listdir(second_subfolder_path)):
                            if 'user' in user_dir:
                                user_id = user_dir
                                user_path = os.path.join(second_subfolder_path, user_dir)
                                
                                # Check if user should be in this room
                                rooms = get_user_rooms(user_id, room_user_mapping)
                                if room not in rooms:
                                    continue
                                
                                # Load data for this session
                                print(f"  Loading {user_id} from {second_subfolder} ({room})...", end=" ")
                                
                                data, label = load_4D_data(
                                    [user_path], 
                                    ALL_MOTION, 
                                    T_MAX, 
                                    second_subfolder, 
                                    user_id
                                )
                                
                                if len(data) == 0:
                                    print(f"  No valid data")
                                    continue
                                
                                print(f"✓ {len(data)} samples")
                                
                                # Initialize storage
                                if user_id not in temp_storage:
                                    temp_storage[user_id] = {}
                                if room not in temp_storage[user_id]:
                                    temp_storage[user_id][room] = []
                                
                                # Store session data
                                temp_storage[user_id][room].append({
                                    'session_name': second_subfolder,
                                    'session_date': session_date,
                                    'data': data,
                                    'labels': label,
                                    'n_samples': len(data)
                                })
    
    print("Sorting sessions chronologically and concatenating...\n")
    for user_id in sorted(temp_storage.keys()):
        fl_clients[user_id] = {}
        
        print(f"── {user_id} ──")
        
        for room in sorted(temp_storage[user_id].keys()):
            # Sort sessions by date
            sessions = temp_storage[user_id][room]
            sessions.sort(key=lambda x: x['session_date'])
            
            print(f"  {room} - Chronological order:")
            for i, session in enumerate(sessions, 1):
                print(f"    {i}. {session['session_date']} - {session['session_name']}: "
                      f"{session['n_samples']} samples")
            
            # Concatenate data chronologically
            all_data = []
            all_labels = []
            session_tags = []
            session_metadata = []
            current_idx = 0
            
            for session in sessions:
                all_data.append(session['data'])
                all_labels.append(session['labels'])
                
                # Tag each sample with its session
                n_samples = session['n_samples']
                session_tags.extend([session['session_name']] * n_samples)
                
                # Store session metadata with indices
                session_metadata.append({
                    'session_name': session['session_name'],
                    'session_date': session['session_date'],
                    'n_samples': n_samples,
                    'start_idx': current_idx,
                    'end_idx': current_idx + n_samples
                })
                current_idx += n_samples
            
            # Store in fl_clients
            fl_clients[user_id][room] = {
                'data': np.concatenate(all_data, axis=0),
                'labels': np.concatenate(all_labels, axis=0),
                'session_tags': np.array(session_tags),
                'sessions': session_metadata
            }
            
            total_samples = len(fl_clients[user_id][room]['data'])
            print(f"  → Total: {total_samples} samples (chronologically ordered)\n")
    

    total_samples = 0
    total_sessions = 0
    
    for user_id in sorted(fl_clients.keys()):
        print(f"Client: {user_id}")
        for room in sorted(fl_clients[user_id].keys()):
            data_shape = fl_clients[user_id][room]['data'].shape
            labels_shape = fl_clients[user_id][room]['labels'].shape
            n_sessions = len(fl_clients[user_id][room]['sessions'])
            
            total_samples += data_shape[0]
            total_sessions += n_sessions
            
            print(f"  {room}: {data_shape[0]} samples from {n_sessions} session(s)")
            print(f"    Data: {data_shape}, Labels: {labels_shape}")
    
    print(f"TOTAL: {total_samples} samples from {total_sessions} sessions across "
          f"{len(fl_clients)} users")

    return fl_clients, data_index

fl_clients, data_index = load_all_user_data(data_bvp, room_user_mapping, subfolder_room_mapping)

In [ ]:

def prepare_grouped_baseline_data(fl_clients, initial_clients, new_clients,
                                   train_ratio=0.70, val_ratio=0.15,
                                   test_ratio=0.15, seed=42):
    
    """
    Three split strategies are applied depending on how many sessions
    a user has across all their rooms in this group:
        >= 3 sessions : session-based  — earlier sessions → train,
                                         middle → val, latest → test
           2 sessions : hybrid         — session 1 → train,
                                         session 2 split 50/50 val/test
           1 session  : stratified     — per-class 70/15/15 split

    """

    # Inner function: split one group
    def split_single_group(group_dict):
        np.random.seed(seed)
        per_user_splits = {}

        # Accumulators for group-level aggregation
        all_train_data,  all_train_labels = [], []
        all_val_data,    all_val_labels   = [], []
        all_test_data,   all_test_labels  = [], []

        # Collect unique users across all rooms in this group
        group_users = set()
        for room_users in group_dict.values():
            group_users.update(room_users)

        for user_id in sorted(group_users):
            if user_id not in fl_clients:
                continue

            # Rooms this user appears in within this group
            user_rooms_in_group = [
                room for room, users in group_dict.items()
                if user_id in users and room in fl_clients[user_id]]

            # Collect all sessions for this user across rooms 
            all_sessions = []
            for room in user_rooms_in_group:
                sessions = fl_clients[user_id][room]['sessions']
                data     = fl_clients[user_id][room]['data']
                labels   = fl_clients[user_id][room]['labels']

                for session in sessions:
                    start_idx = session['start_idx']
                    end_idx   = session['end_idx']
                    all_sessions.append({
                        'session_name': session['session_name'],
                        'session_date': session['session_date'],
                        'room':         room,
                        'data':         data[start_idx:end_idx],
                        'labels':       labels[start_idx:end_idx],
                        'n_samples':    session['n_samples']
                    })

            # Sort sessions chronologically so splits respect time order
            all_sessions.sort(key=lambda x: x['session_date'])
            n_sessions = len(all_sessions)

            for i, session in enumerate(all_sessions, 1):
                print(f"    {i}. {session['session_date']} - {session['session_name']} "
                      f"({session['room']}): {session['n_samples']} samples")

            # STEP 2: Split strategy
            if n_sessions >= 3:
                # Session-based: proportional split by number of sessions
                n_train_sessions = max(1, int(train_ratio * n_sessions))
                n_val_sessions   = max(1, int(val_ratio   * n_sessions))

                train_sessions = all_sessions[:n_train_sessions]
                val_sessions   = all_sessions[n_train_sessions:n_train_sessions + n_val_sessions]
                test_sessions  = all_sessions[n_train_sessions + n_val_sessions:]

                train_data   = np.concatenate([s['data']   for s in train_sessions], axis=0)
                train_labels = np.concatenate([s['labels'] for s in train_sessions], axis=0)

                val_data   = np.concatenate([s['data']   for s in val_sessions],  axis=0) if val_sessions  else np.empty((0, 38, 20, 20, 1))
                val_labels = np.concatenate([s['labels'] for s in val_sessions],  axis=0) if val_sessions  else np.empty((0, 12))

                test_data   = np.concatenate([s['data']   for s in test_sessions], axis=0) if test_sessions else np.empty((0, 38, 20, 20, 1))
                test_labels = np.concatenate([s['labels'] for s in test_sessions], axis=0) if test_sessions else np.empty((0, 12))

                session_info   = {
                    'train_sessions': [s['session_name'] for s in train_sessions],
                    'val_sessions':   [s['session_name'] for s in val_sessions],
                    'test_sessions':  [s['session_name'] for s in test_sessions]
                }
                split_strategy = 'session-based'

            elif n_sessions == 2:
                # first session → train; second session split 50/50 val/test
                train_data   = all_sessions[0]['data'].copy()
                train_labels = all_sessions[0]['labels'].copy()

                second_data   = all_sessions[1]['data']
                second_labels = all_sessions[1]['labels']
                n_val         = len(second_data) // 2

                val_data   = second_data[:n_val].copy()
                val_labels = second_labels[:n_val].copy()
                test_data  = second_data[n_val:].copy()
                test_labels= second_labels[n_val:].copy()

                session_info   = {
                    'train_sessions': [all_sessions[0]['session_name']],
                    'val_sessions':   [all_sessions[1]['session_name']],
                    'test_sessions':  [all_sessions[1]['session_name']]
                }
                split_strategy = 'hybrid-session-sample'

            else:  # n_sessions == 1
                # Stratified: per-class 70/15/15 split
                all_user_data   = np.concatenate([s['data']   for s in all_sessions], axis=0)
                all_user_labels = np.concatenate([s['labels'] for s in all_sessions], axis=0)

                train_parts, train_label_parts = [], []
                val_parts,   val_label_parts   = [], []
                test_parts,  test_label_parts  = [], []

                class_indices  = np.argmax(all_user_labels, axis=1)
                unique_classes = sorted(np.unique(class_indices))
                print(f"  Activities found: {unique_classes} ({len(unique_classes)} total)")

                for class_idx in unique_classes:
                    mask        = class_indices == class_idx
                    class_data  = all_user_data[mask]
                    class_lbls  = all_user_labels[mask]
                    n_c         = len(class_data)
                    n_train     = int(train_ratio * n_c)
                    n_val_c     = int(val_ratio   * n_c)

                    train_parts.append(class_data[:n_train])
                    train_label_parts.append(class_lbls[:n_train])
                    val_parts.append(class_data[n_train:n_train + n_val_c])
                    val_label_parts.append(class_lbls[n_train:n_train + n_val_c])
                    test_parts.append(class_data[n_train + n_val_c:])
                    test_label_parts.append(class_lbls[n_train + n_val_c:])

                train_data   = np.concatenate(train_parts, axis=0)
                train_labels = np.concatenate(train_label_parts, axis=0)
                val_data     = np.concatenate(val_parts,   axis=0)
                val_labels   = np.concatenate(val_label_parts,   axis=0)
                test_data    = np.concatenate(test_parts,  axis=0)
                test_labels  = np.concatenate(test_label_parts,  axis=0)

                # Free intermediate class-split lists immediately
                del train_parts, train_label_parts
                del val_parts,   val_label_parts
                del test_parts,  test_label_parts
                del all_user_data, all_user_labels, class_indices

                session_info   = {
                    'train_sessions': [s['session_name'] for s in all_sessions],
                    'val_sessions':   [s['session_name'] for s in all_sessions],
                    'test_sessions':  [s['session_name'] for s in all_sessions]
                }
                split_strategy = 'stratified'

            # Free all_sessions immediately after splits are built
            del all_sessions
            gc.collect()

            n_total = len(train_data) + len(val_data) + len(test_data)

            # Class distribution check
            train_class_dist = (np.bincount(np.argmax(train_labels, axis=1), minlength=12)
                                if len(train_data) > 0 else np.zeros(12, dtype=int))
            val_class_dist   = (np.bincount(np.argmax(val_labels,   axis=1), minlength=12)
                                if len(val_data)   > 0 else np.zeros(12, dtype=int))
            test_class_dist  = (np.bincount(np.argmax(test_labels,  axis=1), minlength=12)
                                if len(test_data)  > 0 else np.zeros(12, dtype=int))

            if len(train_data) > 0:
                print(f"    Train class dist: {train_class_dist}")
            if len(val_data)   > 0:
                print(f"    Val   class dist: {val_class_dist}")
            if len(test_data)  > 0:
                print(f"    Test  class dist: {test_class_dist}")

            if len(train_data) > 0 and len(val_data) > 0 and len(test_data) > 0:
                train_classes    = set(np.where(train_class_dist > 0)[0])
                val_classes      = set(np.where(val_class_dist   > 0)[0])
                test_classes     = set(np.where(test_class_dist  > 0)[0])
                missing_in_val   = train_classes - val_classes
                missing_in_test  = train_classes - test_classes

            # STEP 5: Store splits
            per_user_splits[user_id] = {
                'train_data':    train_data,
                'train_labels':  train_labels,
                'val_data':      val_data,
                'val_labels':    val_labels,
                'test_data':     test_data,
                'test_labels':   test_labels,
                'total_samples': n_total,
                'train_samples': len(train_data),
                'val_samples':   len(val_data),
                'test_samples':  len(test_data),
                'n_sessions':    n_sessions,
                'split_strategy':split_strategy,
                'rooms':         user_rooms_in_group,
                **session_info}

            # Append references for group aggregation
            all_train_data.append(train_data)
            all_train_labels.append(train_labels)
            all_val_data.append(val_data)
            all_val_labels.append(val_labels)
            all_test_data.append(test_data)
            all_test_labels.append(test_labels)

        # GROUP AGGREGATION
        group_splits = {
            'train_data':   np.concatenate(all_train_data,   axis=0),
            'train_labels': np.concatenate(all_train_labels, axis=0),
            'val_data':     np.concatenate(all_val_data,     axis=0),
            'val_labels':   np.concatenate(all_val_labels,   axis=0),
            'test_data':    np.concatenate(all_test_data,    axis=0),
            'test_labels':  np.concatenate(all_test_labels,  axis=0)}

        del all_train_data,  all_train_labels
        del all_val_data,    all_val_labels
        del all_test_data,   all_test_labels
        gc.collect()

        # Group class distribution
        group_train_dist = np.bincount(
            np.argmax(group_splits['train_labels'], axis=1), minlength=12)
        group_val_dist   = np.bincount(
            np.argmax(group_splits['val_labels'],   axis=1), minlength=12)
        group_test_dist  = np.bincount(
            np.argmax(group_splits['test_labels'],  axis=1), minlength=12)


        group_train_classes = set(np.where(group_train_dist > 0)[0])
        group_val_classes   = set(np.where(group_val_dist   > 0)[0])
        group_test_classes  = set(np.where(group_test_dist  > 0)[0])

        if group_train_classes == group_val_classes == group_test_classes:
            print(f"\n  All splits have complete class coverage in this group")
        else:
            missing_in_val  = group_train_classes - group_val_classes
            missing_in_test = group_train_classes - group_test_classes
            if missing_in_val:
                print(f"\n  Group classes in train but NOT in val: {missing_in_val}")
            if missing_in_test:
                print(f"  Group classes in train but NOT in test: {missing_in_test}")

        return per_user_splits, group_splits
    
    # Run split for both groups 
    group1_per_user, group1_splits = split_single_group(
        initial_clients)
    group2_per_user, group2_splits = split_single_group(
        new_clients)

    return group1_per_user, group1_splits, group2_per_user, group2_splits



def get_data_split_for_seed(seed, fl_clients, 
                            train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                            case='case1'):
    """
    Three cases define different G1/G2 compositions to study how new clients
    with different data heterogeneity levels join an existing federation:
        case1 : G1 = single-session users (LOW severity)
                G2 = multi-session users  (HIGH/MOD/CRIT severity)
        case2 : G1 = multi-session users  (HIGH/MOD/CRIT severity)  [swapped from case1]
                G2 = single-session users (LOW severity)
        case3 : G1 and G2 both contain mixed-severity users
    """

    if case == 'case1':
        # Case 1: Imbalanced users (G2) join balanced users (G1)
        initial_clients = {
            'room1': ['user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
            'room2': ['user4', 'user6'],
            'room3': ['user7', 'user8', 'user9']
        }
        new_clients = {
            'room1': ['user1', 'user2', 'user3', 'user5'],
            'room2': ['user1', 'user2', 'user3', 'user5'],
            'room3': ['user3']
        }
        
    elif case == 'case2':
        # Case 2: Balanced users (G2) join imbalanced users (G1)
        # (Swapped from case1)
        initial_clients = {
            'room1': ['user1', 'user2', 'user3', 'user5'],
            'room2': ['user1', 'user2', 'user3', 'user5'],
            'room3': ['user3']
        }
        new_clients = {
            'room1': ['user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
            'room2': ['user4', 'user6'],
            'room3': ['user7', 'user8', 'user9']
        }
        
    elif case == 'case3':
        # Case 3: Mixed severity in both groups
        initial_clients = {
            'room1': ['user1', 'user2', 'user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
            'room2': ['user1', 'user2', 'user4', 'user6']
        }
        new_clients = {
            'room1': ['user3', 'user5'],
            'room2': ['user3', 'user5'],
            'room3': ['user3', 'user7', 'user8', 'user9']
        }
        
    else:
        raise ValueError(f"Unknown case: {case}. Must be 'case1', 'case2', or 'case3'")
    

    # VALIDATE: No user overlap between groups
    g1_users = set()
    for room_users in initial_clients.values():
        g1_users.update(room_users)
    
    g2_users = set()
    for room_users in new_clients.values():
        g2_users.update(room_users)
    
    overlap = g1_users & g2_users
    if overlap:
        raise ValueError(f"❌ Invalid configuration for {case}: "
                        f"Users {overlap} appear in both groups!")
    

    print(f"  CASE: {case.upper()}")
    print(f"  G1 (Initial) users: {sorted(g1_users)}")
    print(f"  G2 (New) users:     {sorted(g2_users)}")
    print(f"  No overlap: ")
    

    # PREPARE DATA SPLITS
    group1_per_user, group1_splits, \
    group2_per_user, group2_splits = prepare_grouped_baseline_data(
        fl_clients=fl_clients,
        initial_clients=initial_clients,
        new_clients=new_clients,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed)
    
    return {
        'group1_per_user': group1_per_user,
        'group1_splits':   group1_splits,
        'group2_per_user': group2_per_user,
        'group2_splits':   group2_splits,
        'case':            case,
        'g1_users':        sorted(g1_users),
        'g2_users':        sorted(g2_users)
    }

# Memory-efficient data loader that holds splits for only one seed at a time
class LazySeedDataLoader:

    def __init__(self, fl_clients, case='case1',
                 train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
        self.fl_clients  = fl_clients
        self.case        = case
        self.train_ratio = train_ratio
        self.val_ratio   = val_ratio
        self.test_ratio  = test_ratio
        self._current_seed = None
        self._current_data = None
        
        # Validate case
        valid_cases = ['case1', 'case2', 'case3']
        if case not in valid_cases:
            raise ValueError(f"Invalid case: {case}. Must be one of {valid_cases}")
        

    def load(self, seed):
        """Load and cache splits for the given seed. Reuses cache if same seed."""
        if self._current_seed == seed and self._current_data is not None:
            print(f"  Seed {seed} already loaded — reusing")
            return self._current_data

        # Free previous data if exists
        if self._current_data is not None:
            self._free_current()

        # Load new data
        print(f"  Loading data for seed {seed} ({self.case})...")
        self._current_data = get_data_split_for_seed(
            seed=seed,
            fl_clients=self.fl_clients,
            train_ratio=self.train_ratio,
            val_ratio=self.val_ratio,
            test_ratio=self.test_ratio,
            case=self.case)  # ← Pass case parameter
        
        self._current_seed = seed

        # Print summary
        g1 = self._current_data['group1_per_user']
        g2 = self._current_data['group2_per_user']
        
        g1_train = sum(len(s['train_data']) for s in g1.values())
        g2_train = sum(len(s['train_data']) for s in g2.values())
        
        print(f"  Seed {seed} loaded:")
        print(f"     G1 users : {sorted(g1.keys())}")
        print(f"     G2 users : {sorted(g2.keys())}")
        print(f"     G1 train : {g1_train} samples")
        print(f"     G2 train : {g2_train} samples")

        return self._current_data

    def _free_current(self):
        """Explicitly delete all numpy arrays from the current seed to free RAM."""
        
        if self._current_data is None:
            return

        freed_seed = self._current_seed

        # Free per-user splits
        for group_key in ['group1_per_user', 'group2_per_user']:
            per_user = self._current_data.get(group_key, {})
            for uid, splits in per_user.items():
                for arr_key in ['train_data', 'train_labels',
                               'val_data',   'val_labels',
                               'test_data',  'test_labels']:
                    if arr_key in splits:
                        del splits[arr_key]

        # Free group aggregated splits
        for group_key in ['group1_splits', 'group2_splits']:
            splits = self._current_data.get(group_key, {})
            for arr_key in ['train_data', 'train_labels',
                           'val_data',   'val_labels',
                           'test_data',  'test_labels']:
                if arr_key in splits:
                    del splits[arr_key]

        del self._current_data
        self._current_data = None
        self._current_seed = None

        gc.collect()
        print(f"  Freed data for seed {freed_seed}")

    def __del__(self):
        """Ensure memory is released when the loader object is garbage collected."""
        self._free_current()


In [ ]:
# DataGenerator for FedPer local training
class FedPerDataGenerator(tf.keras.utils.Sequence):

    def __init__(self, data, labels, batch_size, shuffle=True):
        self.data       = data    
        self.labels     = labels  
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.indexes    = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start         = index * self.batch_size
        end           = min(start + self.batch_size, len(self.data))
        batch_indexes = self.indexes[start:end]
        return self.data[batch_indexes], self.labels[batch_indexes]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

#  Data Generator for inference / evaluation 
class FedPerPredictGenerator(tf.keras.utils.Sequence):

    def __init__(self, data, batch_size=32):
        self.data       = data
        self.batch_size = batch_size
        self.indexes    = np.arange(len(data))

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end   = min(start + self.batch_size, len(self.data))
        return self.data[self.indexes[start:end]]


# Evaluation using PredictGenerator
def evaluate_with_generator(model, test_data, test_labels, batch_size=32):

    try:
        pred_gen     = FedPerPredictGenerator(test_data, batch_size)
        predictions  = model.predict(pred_gen, verbose=0)
        pred_classes = np.argmax(predictions,  axis=1)
        true_classes = np.argmax(test_labels,  axis=1)
        return float(np.mean(pred_classes == true_classes))

    except tf.errors.ResourceExhaustedError:
        # Retry with smaller batch if GPU memory is exhausted
        try:
            pred_gen     = FedPerPredictGenerator(test_data, batch_size=8)
            predictions  = model.predict(pred_gen, verbose=0)
            pred_classes = np.argmax(predictions,  axis=1)
            true_classes = np.argmax(test_labels,  axis=1)
            return float(np.mean(pred_classes == true_classes))
        except Exception as e:
            print(f"    Retry failed: {e}")
            return 0.0

    except Exception as e:
        print(f"    Eval error: {e}")
        return 0.0
    

# CUSTOM DATA GENERATOR FOR Personalized MODELS
class FedPerDualHeadGenerator(tf.keras.utils.Sequence):

    def __init__(self, data, labels, batch_size, shuffle=True):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.data))
        batch_indexes = self.indexes[start:end]
        batch_x = self.data[batch_indexes]
        batch_y = self.labels[batch_indexes]
        return batch_x, {'global_output': batch_y, 'personal_output': batch_y}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
# Add Memory Management Utilities
# Pre-convert reference weights ONCE (not every round/batch)
def create_reference_weights_numpy(model):
    return [w.copy() for w in model.get_weights()]
        
# Aggressively clear GPU memory
def clear_gpu_memory():
    gc.collect()
    try:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# Safely delete a model and free its memory
def safe_model_delete(model):
    if model is None:
        return
    try:
        # Clear sub-models first (dual-head case)
        if hasattr(model, 'base_model'):
            try:
                del model.base_model
            except Exception:
                pass

        # Release layer references
        if hasattr(model, 'layers'):
            for layer in model.layers:
                try:
                    # Clear layer weights from memory
                    if hasattr(layer, '_trainable_weights'):
                        layer._trainable_weights.clear()
                    if hasattr(layer, '_non_trainable_weights'):
                        layer._non_trainable_weights.clear()
                except Exception:
                    pass

        del model
    except Exception:
        pass
    gc.collect()

In [ ]:
# Loss Functions
# Example usage for Federated Learning
input_shape = (38, 20, 20, 1)
n_class = 12
# Weighted Loss for Class Imbalance
def get_class_weights(train_labels, epsilon=1e-8):
    class_counts = np.sum(train_labels, axis=0)
    n_samples = np.sum(class_counts)
    active_mask = class_counts > 0
    n_active = np.sum(active_mask)

    weights = np.zeros(len(class_counts))
    for c in range(len(class_counts)):
        if class_counts[c] > 0:
            weights[c] = n_samples / (n_active * class_counts[c])
        else:
            weights[c] = epsilon

    print(f" Class weights: {np.round(weights, 2)}")
    return weights

# Label smoothing for low severity users
def weighted_loss_with_smoothing(class_weights, smoothing=0.1):
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        n_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_smooth = y_true * (1.0 - smoothing) + (smoothing / n_classes)
        y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        per_class_loss = -y_true_smooth * tf.math.log(y_pred_clipped)  # (B, C)
        weighted_loss = per_class_loss * class_weights_tensor           # (B, C)
        return tf.reduce_mean(tf.reduce_sum(weighted_loss, axis=-1))
    return loss_fn

# Adaptive Label Smoothing Per User
def get_adaptive_smoothing(train_labels, n_total_classes=12, base_smoothing=0.1):
    class_counts  = np.sum(train_labels, axis=0)
    active_classes = np.sum(class_counts > 0)
    active_ratio   = active_classes / n_total_classes
    smoothing      = round(base_smoothing * active_ratio, 4)
    
    print(f"    Active classes: {int(active_classes)}/{n_total_classes} "
          f"→ label_smoothing={smoothing:.4f}")
    return smoothing


# HELPER FUNCTIONS FOR DUAL-HEAD
def sync_full_model_weights(local_model, global_model):
    synced = 0
    for layer in global_model.layers:
        try:
            local_layer = local_model.get_layer(layer.name)
            ws = layer.get_weights()
            local_ws = local_layer.get_weights()
            if ws and local_ws and [w.shape for w in ws] == [w.shape for w in local_ws]:
                local_layer.set_weights(ws)
                synced += 1
        except (ValueError, IndexError):
            pass
    return synced

# Extract weights from the 'global_output' Dense layer.
def get_global_output_weights(model):
    try:
        layer = model.get_layer('global_output')
        ws = layer.get_weights()
        return [w.copy() for w in ws] if ws else None
    except ValueError:
        return None

# Set weights of the 'global_output' Dense layer
def set_global_output_weights(model, weights):
    """Only applies if the layer exists and shapes match exactly."""
    try:
        layer = model.get_layer('global_output')
        current = layer.get_weights()
        if current and [w.shape for w in current] == [w.shape for w in weights]:
            layer.set_weights(weights)
            return True
    except ValueError:
        pass
    return False



# Unified hybrid loss for ALL users (both HIGH and LOW severity)
# HIGH-severity users (user1,2,3,5,6): stronger regularization
# LOW-severity users (user4,7-17): lighter regularization
class AdaptiveDASPRLoss(tf.keras.losses.Loss):
    """
    Unified hybrid loss for users with LOW data heterogeneity severity.

    Combines three components, all auto-scaled from the train/val
    class distribution at construction time:

        1. Focal loss  — down-weights easy examples; class-weighted
                         and label-smoothed to handle imbalance
        2. Confidence penalty — penalises over-confidence on seen classes
                                to keep the model calibrated
        3. Entropy regularisation — encourages uniform predictions over
                                    classes unseen during training,
                                    reducing catastrophic forgetting
    """
    
    def __init__(self, train_labels, val_labels, n_class=12,
                 focal_gamma=2.0, severity_override=None,
                 name='adaptive_daspr_loss'):
        super().__init__(name=name)
        
        self.n_class = n_class
        
        train_class_counts = np.sum(train_labels, axis=0)
        val_class_counts = np.sum(val_labels, axis=0)
        
        train_classes = set(np.where(train_class_counts > 0)[0])
        val_classes = set(np.where(val_class_counts > 0)[0])
        overlap_classes = train_classes & val_classes
        unseen_in_train = val_classes - train_classes
        
        train_coverage = len(train_classes) / n_class
        val_coverage = len(val_classes) / n_class
        
        if len(train_classes) > 0 and len(val_classes) > 0:
            overlap_ratio = len(overlap_classes) / max(len(train_classes), len(val_classes))
        else:
            overlap_ratio = 1.0
        
        self.overlap_ratio = overlap_ratio
        self.n_unseen = len(unseen_in_train)
        self.n_train_classes = len(train_classes)
        self.train_coverage = train_coverage
        self.val_coverage = val_coverage
        
        # Class weights
        class_weights = np.zeros(n_class, dtype=np.float32)
        n_samples = np.sum(train_class_counts)
        n_active = np.sum(train_class_counts > 0)
        for c in range(n_class):
            if train_class_counts[c] > 0:
                class_weights[c] = n_samples / (n_active * train_class_counts[c])
            else:
                class_weights[c] = 0.1
        self.class_weights_tf = tf.constant(class_weights, dtype=tf.float32)
        
        # Masks
        seen_mask = np.zeros(n_class, dtype=np.float32)
        seen_mask[list(train_classes)] = 1.0
        self.seen_mask_tf = tf.constant(seen_mask, dtype=tf.float32)
        
        unseen_mask = np.zeros(n_class, dtype=np.float32)
        if unseen_in_train:
            unseen_mask[list(unseen_in_train)] = 1.0
        self.unseen_mask_tf = tf.constant(unseen_mask, dtype=tf.float32)
        
        # Auto-compute parameters
        active_counts = train_class_counts[train_class_counts > 0]
        if len(active_counts) > 1:
            normalized_imbalance = min(float(np.max(active_counts) / (np.min(active_counts) + 1e-8)) / 10.0, 1.0)
        else:
            normalized_imbalance = 0.0
        
        # Label smoothing
        self.label_smoothing = float(np.clip(
            0.05 + 0.15 * (1.0 - overlap_ratio) + 0.05 * normalized_imbalance + 0.05 * (1.0 - train_coverage),
            0.03, 0.25))
        
        # Focal gamma
        self.focal_gamma = float(np.clip(1.5 + 1.0 * normalized_imbalance + 0.5 * (1.0 - overlap_ratio), 1.0, 3.0))
        
        # Entropy regularization
        if self.n_unseen > 0:
            unseen_ratio = self.n_unseen / n_class
            self.lambda_entropy = float(np.clip(0.05 + 0.30 * unseen_ratio + 0.15 * (1.0 - overlap_ratio), 0.03, 0.40))
        else:
            self.lambda_entropy = float(np.clip(0.03 + 0.10 * (1.0 - overlap_ratio), 0.02, 0.15))
        
        # Confidence
        self.lambda_confidence = float(np.clip(
            0.02 + 0.10 * (1.0 - overlap_ratio) + 0.05 * (1.0 - val_coverage) + 0.03 * normalized_imbalance,
            0.02, 0.15))
        
        self.target_confidence = float(np.clip(
            0.5 + 0.3 * overlap_ratio + 0.1 * train_coverage - 0.15 * (self.n_unseen / n_class),
            0.30, 0.90))
        
        print(f"    AdaptiveDASPRLoss [LOW] (AUTO-COMPUTED):")
        print(f"      overlap={overlap_ratio:.2f} smoothing={self.label_smoothing:.3f} "
              f"gamma={self.focal_gamma:.2f} entropy={self.lambda_entropy:.3f}")
    
    def call(self, y_true, y_pred):
        n_classes_tf = tf.cast(self.n_class, tf.float32)
        y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Focal loss with class weighting + label smoothing
        y_true_smooth = (y_true * (1.0 - self.label_smoothing) + 
                        (self.label_smoothing / n_classes_tf))
        per_class_loss = -y_true_smooth * tf.math.log(y_pred_clipped)
        weighted_loss = per_class_loss * self.class_weights_tf
        ce_per_sample = tf.reduce_sum(weighted_loss, axis=-1)
        p_t = tf.reduce_sum(y_true * y_pred_clipped, axis=-1)
        focal_weight = tf.pow(1.0 - p_t, self.focal_gamma)
        focal_loss = tf.reduce_mean(focal_weight * ce_per_sample)
        
        # Confidence calibration
        max_pred_seen = tf.reduce_max(y_pred_clipped * self.seen_mask_tf, axis=-1)
        confidence_excess = tf.nn.relu(max_pred_seen - self.target_confidence)
        confidence_penalty = tf.reduce_mean(confidence_excess ** 2)
        
        # Entropy regularization for unseen classes
        if self.n_unseen > 0:
            y_pred_unseen = y_pred_clipped * self.unseen_mask_tf
            y_pred_unseen_sum = tf.reduce_sum(y_pred_unseen, axis=-1, keepdims=True)
            y_pred_unseen_norm = y_pred_unseen / (y_pred_unseen_sum + 1e-8)
            entropy_unseen = -tf.reduce_sum(
                y_pred_unseen_norm * tf.math.log(y_pred_unseen_norm + 1e-8), axis=-1)
            max_entropy = tf.math.log(tf.cast(self.n_unseen, tf.float32))
            entropy_penalty = tf.reduce_mean(max_entropy - entropy_unseen)
        else:
            entropy_penalty = tf.constant(0.0, dtype=tf.float32)
        
        total_loss = (focal_loss + 
                     self.lambda_confidence * confidence_penalty +
                     self.lambda_entropy * entropy_penalty)
        return total_loss

    def get_config(self):
        config = super().get_config()
        config.update({'n_class': self.n_class})
        return config


In [ ]:
# Score a user's data heterogeneity and assign a severity level.
def detect_severity_distribution_only(train_labels, val_labels, n_class=12):
    train_counts = np.sum(train_labels, axis=0)
    val_counts = np.sum(val_labels, axis=0)
    train_classes = set(np.where(train_counts > 0)[0])
    val_classes = set(np.where(val_counts > 0)[0])
    overlap_classes = train_classes & val_classes

    train_coverage = len(train_classes) / n_class
    val_coverage = len(val_classes) / n_class

    if len(train_classes) > 0 and len(val_classes) > 0:
        train_to_val_overlap = len(overlap_classes) / len(train_classes)
        val_to_train_overlap = len(overlap_classes) / len(val_classes)
        overlap_ratio = min(train_to_val_overlap, val_to_train_overlap)
    else:
        train_to_val_overlap = 1.0
        val_to_train_overlap = 1.0
        overlap_ratio = 1.0

    train_dist = train_counts / (train_counts.sum() + 1e-8)
    val_dist = val_counts / (val_counts.sum() + 1e-8)
    kl_divergence = np.sum(val_dist * np.log((val_dist + 1e-8) / (train_dist + 1e-8)))

    severity_score = 0.0
    if train_coverage < 0.5:     severity_score += 20
    elif train_coverage < 0.7:   severity_score += 10
    elif train_coverage < 0.9:   severity_score += 5
    if overlap_ratio < 0.3:      severity_score += 35
    elif overlap_ratio < 0.5:    severity_score += 25
    elif overlap_ratio < 0.7:    severity_score += 15
    elif overlap_ratio < 0.9:    severity_score += 5
    if kl_divergence > 2.0:      severity_score += 25
    elif kl_divergence > 1.0:    severity_score += 15
    elif kl_divergence > 0.5:    severity_score += 8
    if val_coverage < 0.3:       severity_score += 15
    elif val_coverage < 0.5:     severity_score += 8

    if severity_score >= 70:     severity = 'CRITICAL'
    elif severity_score >= 50:   severity = 'HIGH'
    elif severity_score >= 30:   severity = 'MODERATE'
    else:                        severity = 'LOW'

    confidence = min(1.0, severity_score / 100.0) if severity != 'LOW' else 1.0 - (severity_score / 100.0)

    return severity, {
        'severity': severity, 'confidence': confidence, 'score': severity_score,
        'overlap_ratio': overlap_ratio, 'train_to_val_overlap': train_to_val_overlap,
        'val_to_train_overlap': val_to_train_overlap, 'train_coverage': train_coverage,
        'val_coverage': val_coverage, 'kl_divergence': kl_divergence,
        'n_train_classes': len(train_classes), 'n_val_classes': len(val_classes),
        'n_overlap_classes': len(overlap_classes),
    }


def get_user_domain(user_id):
    """
    Used to label personal head layers with a domain-specific name
    so that different users' personal layers do not share names
    when loaded into the same session.
    """
    mapping = {
        'user1': 'room1', 'user2': 'room1', 'user3': 'room1',
        'user4': 'room2', 'user5': 'room2', 'user6': 'room2',
        'user7': 'room3', 'user8': 'room3', 'user9': 'room3',
        'user10': 'room1', 'user11': 'room1', 'user12': 'room1',
        'user13': 'room1', 'user14': 'room1', 'user15': 'room1',
        'user16': 'room1', 'user17': 'room1',
    }
    return mapping.get(user_id, 'room1')


In [ ]:
# Layers to be shared
BACKBONE_LAYER_NAMES = ['bb_conv', 'bb_td_dense', 'bb_bigru', 'bb_lstm']
# Severity levels that use personalized models
PERSONAL_SEVERITIES  = ['LOW']
# Severity levels that use the shared global model
GLOBAL_SEVERITIES    = ['MODERATE', 'HIGH', 'CRITICAL']

# Build the shared feature extraction backbone used by all client models
def shared_backbone(input_shape):
    model_input = Input(shape=input_shape, dtype='float32', name='bb_input')
    x = TimeDistributed(Conv2D(16, kernel_size=(5, 5), activation='relu',data_format='channels_last'),name='bb_conv')(model_input)
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)),name='bb_pool')(x)
    x = TimeDistributed(Flatten(), name='bb_td_flatten')(x)
    x = TimeDistributed(Dense(128, activation='gelu',kernel_regularizer=tf.keras.regularizers.l2(0.01)),name='bb_td_dense')(x)
    x = Bidirectional(GRU(128, return_sequences=True,kernel_regularizer=tf.keras.regularizers.l2(0.05)),name='bb_bigru')(x)
    x = Dropout(0.2, name='bb_drop1')(x)
    x = LSTM(128, return_sequences=True,kernel_regularizer=tf.keras.regularizers.l2(0.01),name='bb_lstm')(x)
    x = Dropout(0.2, name='bb_drop2')(x)
    return model_input, x


def compile_global_model(input_shape, n_class, class_weights=None):
    shared_input, shared_output = shared_backbone(input_shape)
    x  = Flatten(name='bb_flatten')(shared_output)
    model_output = Dense(n_class, activation='softmax',name='global_output')(x)
    global_model = Model(inputs=shared_input, outputs=model_output)

    if class_weights is not None:
        loss_fn = weighted_loss_with_smoothing(class_weights, smoothing=0.05)
    else:
        loss_fn = CategoricalCrossentropy(label_smoothing=0.05)

    global_model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
        loss=loss_fn,
        metrics=['accuracy'])

    del loss_fn
    gc.collect()
    return global_model

# Extract a copy of the backbone layer weights from a model
def get_backbone_weights(model):
    backbone_weights = {}
    for layer_name in BACKBONE_LAYER_NAMES:
        try:
            layer   = model.get_layer(layer_name)
            weights = layer.get_weights()
            if weights:
                backbone_weights[layer_name] = [w.copy() for w in weights]
        except ValueError:
            continue
    return backbone_weights

# Apply backbone weights to a model's matching layers
def set_backbone_weights(model, backbone_weights):
    updated = 0
    for layer_name, weights in backbone_weights.items():
        try:
            layer   = model.get_layer(layer_name)
            current = layer.get_weights()
            if (current and
                    [w.shape for w in current] == [w.shape for w in weights]):
                layer.set_weights(weights)
                updated += 1
        except ValueError:
            continue
    return updated

# Compile a personalized model with separate loss weights per output head
def _compile_dual_head_model(model, loss_fn,
                              global_weight=0.3, personal_weight=0.7):
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(0.001),
        loss={'global_output':   loss_fn,'personal_output': loss_fn},
        loss_weights={'global_output':   global_weight,'personal_output': personal_weight},
        metrics={'personal_output': 'accuracy'})
    model._custom_loss_weights = {'global_output':   global_weight, 'personal_output': personal_weight}
    return model

# Build and select the best personalized model architecture for a LOW-severity
def _create_dual_head_model(input_shape, n_class, domain, n_samples,
                             global_model, loss_fn, val_data, val_labels,
                             threshold=750, num_trials=3):
    use_complex = (n_samples or 0) > threshold

    if use_complex:
        max_units    = min(256, (n_samples or 512) // 4)
        units_pool_1 = list(range(32, max_units + 1, 32))
        units_pool_2 = {
            'room1': [64, 128, 256],
            'room2': [32, 64, 128],
            'room3': [16, 32,  64]}
    else:
        max_units    = min(96, max(32, (n_samples or 256) // 6))
        step         = max(32, max_units // 4)
        units_pool_1 = list(range(32, max_units + 1, step))
        if len(units_pool_1) < 2:
            units_pool_1 = [32, 64, max_units]
        units_pool_2 = None

    best_model = None
    best_acc   = -1.0

    for trial in range(num_trials):
        model = None
        try:
            gc.collect()
            # Build backbone + global head (shared across all trials)
            t_in, t_out = shared_backbone(input_shape)
            x_flat      = Flatten(name='bb_flatten')(t_out)
            global_out  = Dense(n_class, activation='softmax',
                                name='global_output')(x_flat)

            if use_complex and units_pool_2:
                d1     = units_pool_1[trial % len(units_pool_1)]
                x_pers = Dense(d1, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.01), name=f'personal_dense_1_{domain}')(x_flat)
                p2 = units_pool_2.get(domain, [32, 64, 128])
                d2     = p2[trial % len(p2)]
                x_pers = Dense(d2, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05), name=f'personal_dense_2_{domain}')(x_pers)
                print(f" Trial {trial+1}: personal=({d1},{d2})")
            else:
                d1     = units_pool_1[trial % len(units_pool_1)]
                x_pers = Dense(d1, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.01), name=f'personal_dense_1_{domain}')(x_flat)
                print(f" Trial {trial+1}: personal=({d1})")

            personal_out = Dense(n_class, activation='softmax', name='personal_output')(x_pers)
            model = Model(inputs=t_in,
                          outputs={'global_output':   global_out,
                                   'personal_output': personal_out})
            # Initialise backbone from global model
            sync_full_model_weights(model, global_model)
            _compile_dual_head_model(model, loss_fn)

            if val_data is not None and val_labels is not None:
                gen            = FedPerPredictGenerator(val_data, batch_size=32)
                preds          = model.predict(gen, verbose=0)
                del gen                                    

                personal_preds = preds['personal_output'].copy()
                del preds                                 
                gc.collect()

                acc     = float(np.mean(
                    np.argmax(personal_preds, 1) ==
                    np.argmax(val_labels,     1)))
                del personal_preds                       
                is_best = acc > best_acc

                print(f"      → personal_acc={acc:.4f}"
                      f"{'  ← BEST' if is_best else ''}")

                if is_best:
                    if best_model is not None:
                        safe_model_delete(best_model)
                        gc.collect()
                    best_acc   = acc
                    best_model = model
                else:
                    safe_model_delete(model)
                    model = None
                    gc.collect()
            else:
                best_model = model
                break

        except Exception as e:
            print(f" Trial {trial+1} failed: {e}")
            if model is not None:
                safe_model_delete(model)
                model = None
            gc.collect()

    # Fallback:  all trials failed — use minimal architecture
    if best_model is None:
        print(" All trials failed — using fallback model")
        f_in, f_out  = shared_backbone(input_shape)
        x_flat       = Flatten(name='bb_flatten')(f_out)
        global_out   = Dense(n_class, activation='softmax', name='global_output')(x_flat)
        x_pers       = Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01), name=f'personal_dense_1_{domain}')(x_flat)
        personal_out = Dense(n_class, activation='softmax', name='personal_output')(x_pers)
        best_model   = Model(inputs=f_in,
                             outputs={'global_output':   global_out,
                                      'personal_output': personal_out})
        sync_full_model_weights(best_model, global_model)
        _compile_dual_head_model(best_model, loss_fn)

    print(f" Dual-head model: personal_acc={best_acc:.4f}")
    return best_model


# Create the appropriate model type for a user based on their severity level
def create_model_for_user(user_id, input_shape, n_class, severity,
                           train_labels, val_labels, val_data,
                           global_model, n_samples, domain,
                           threshold=750, num_trials=3):

    print(f"\n  Creating model for {user_id} [{severity}]...")

    if severity in PERSONAL_SEVERITIES:
        loss_fn = AdaptiveDASPRLoss(
            train_labels, val_labels, n_class, severity_override=severity)
        model = _create_dual_head_model(
            input_shape, n_class, domain, n_samples,
            global_model, loss_fn, val_data, val_labels,
            threshold, num_trials)
        del loss_fn    
        gc.collect()

    else:
        shared_input, shared_output = shared_backbone(input_shape)
        x          = Flatten(name='bb_flatten')(shared_output)
        global_out = Dense(n_class, activation='softmax',
                           name='global_output')(x)
        model      = Model(inputs=shared_input, outputs=global_out)
        sync_full_model_weights(model, global_model)
        model.compile(
            optimizer=tf.keras.optimizers.RMSprop(0.001),
            loss=CategoricalCrossentropy(label_smoothing=0.05),
            metrics=['accuracy'])
        print(f" High/Critical/Moderate model [{severity}] (simple CE)")

    return model

In [ ]:
# Adaptive controller 
# Maximum number of historical values to retain per user
MAX_HISTORY = 100

# Per-client adaptive proximal regularisation strength for FedProx
class AdaptiveMuController:
    def __init__(self, user_ids,
                 mu_min        = 0.001,
                 mu_max        = 0.5,
                 mu_init       = 0.01,
                 total_rounds  = 50,
                 schedule_mode = 'cosine',
                 divergence_w  = 0.4,
                 performance_w = 0.3,
                 ema_alpha     = 0.3):

        self.user_ids      = user_ids
        self.mu_min        = mu_min
        self.mu_max        = mu_max
        self.mu_init       = mu_init
        self.total_rounds  = total_rounds
        self.schedule_mode = schedule_mode
        self.divergence_w  = divergence_w
        self.performance_w = performance_w
        self.ema_alpha     = ema_alpha

        self.current_mu         = {uid: mu_init for uid in user_ids}
        self.mu_history         = {uid: [mu_init] for uid in user_ids}
        self.div_history        = {uid: []        for uid in user_ids}
        self.acc_history        = {uid: []        for uid in user_ids}
        self.prev_local_weights = {uid: None      for uid in user_ids}

    # Return a round-progress-based scaling factor in [0.5, 1.5]
    def _schedule_factor(self, round_num):
        progress = round_num / max(self.total_rounds - 1, 1)

        if self.schedule_mode == 'cosine':
            factor = 1.0 + 0.5 * np.sin(np.pi * progress)
        elif self.schedule_mode == 'linear':
            factor = 0.5 + progress
        elif self.schedule_mode == 'warmup_then_increase':
            warmup = 0.2
            if progress < warmup:
                factor = 0.5
            else:
                factor = 0.5 + (progress - warmup) / (1.0 - warmup)
        else:
            factor = 1.0

        return float(np.clip(factor, 0.5, 1.5))
    
    # Return a factor in [0.7, 1.5] based on weight divergence from global
    def _divergence_factor(self, uid, global_weights):
        if self.prev_local_weights[uid] is None:
            return 1.0

        prev_w   = self.prev_local_weights[uid]
        global_w = global_weights

        if len(prev_w) != len(global_w):
            return 1.0
        
        # Compute mean squared parameter difference
        total_sq_diff = 0.0
        total_params  = 0
        for pw, gw in zip(prev_w, global_w):
            if pw.shape == gw.shape:
                total_sq_diff += np.sum((pw - gw) ** 2)
                total_params  += pw.size

        if total_params == 0:
            return 1.0

        norm_div = total_sq_diff / total_params
        self.div_history[uid].append(float(norm_div))

        if len(self.div_history[uid]) > MAX_HISTORY:
            self.div_history[uid] = self.div_history[uid][-MAX_HISTORY:]
        
        # tanh maps divergence to a smooth factor in [0.7, 1.5]
        factor = 0.7 + (0.8 * np.tanh(norm_div / 0.01))
        return float(np.clip(factor, 0.7, 1.5))
    
    # Return a factor in [0.6, 1.3] based on validation accuracy
    def _performance_factor(self, uid, val_acc):
        if val_acc is None:
            return 1.0

        self.acc_history[uid].append(val_acc)

        if len(self.acc_history[uid]) > MAX_HISTORY:
            self.acc_history[uid] = self.acc_history[uid][-MAX_HISTORY:]

        if val_acc < 0.5:
            factor = 0.6
        elif val_acc < 0.7:
            factor = 0.6 + 0.4 * ((val_acc - 0.5) / 0.2)
        else:
            factor = 1.0 + 0.3 * min((val_acc - 0.7) / 0.3, 1.0)

        return float(np.clip(factor, 0.6, 1.3))
    
    # Compute and store the updated proximal term for a user at the current round
    def update_mu(self, round_num, uid,
                  global_weights, val_acc=None,
                  verbose=True):
        s_schedule    = self._schedule_factor(round_num)
        s_divergence  = self._divergence_factor(uid, global_weights)
        s_performance = self._performance_factor(uid, val_acc)

        schedule_w      = 1.0 - self.divergence_w - self.performance_w
        combined_factor = (schedule_w          * s_schedule   +
                           self.divergence_w   * s_divergence +
                           self.performance_w  * s_performance)

        mu_raw    = self.mu_init * combined_factor
        mu_prev   = self.current_mu[uid]
        # EMA smoothing to avoid abrupt μ changes
        mu_smooth = (self.ema_alpha       * mu_raw  +
                     (1 - self.ema_alpha) * mu_prev)
        mu_new    = float(np.clip(mu_smooth, self.mu_min, self.mu_max))
        mu_new    = round(mu_new, 5)

        self.current_mu[uid] = mu_new
        self.mu_history[uid].append(mu_new)

        if verbose:
            print(f"  μ update [{uid}]: "
                  f"sched={s_schedule:.3f} "
                  f"div={s_divergence:.3f} "
                  f"perf={s_performance:.3f} "
                  f"→ μ={mu_prev:.5f} → {mu_new:.5f}")
        return mu_new
    
    # Save model weights for divergence tracking
    def save_local_weights(self, uid, local_model):
        if self.prev_local_weights[uid] is not None:
            del self.prev_local_weights[uid]
            gc.collect()

        self.prev_local_weights[uid] = [
            w.copy() for w in local_model.get_weights()
        ]

    def round_summary(self, round_num):
        mu_vals = [self.mu_history[uid][-1]
                   for uid in self.user_ids
                   if self.mu_history[uid]]
        print(f"\n  Adaptive μ Summary — Round {round_num + 1}:")
        print(f"    Range : {min(mu_vals):.5f} – {max(mu_vals):.5f}")
        print(f"    Mean  : {np.mean(mu_vals):.5f}")
        print(f"    Std   : {np.std(mu_vals):.5f}")
        for uid in self.user_ids:
            if self.mu_history[uid]:
                print(f"    {uid:<10}: μ={self.current_mu[uid]:.5f}")

# Compute the output-layer proximal regularisation strength for a client
def compute_output_prox_strength(client_severity, overlap_ratio, val_coverage,
                                  train_coverage, round_num=0, total_rounds=50):
    if client_severity in PERSONAL_SEVERITIES:
        return 0.0

    base_strength = {
        'CRITICAL': 0.50,
        'HIGH':     0.30,
        'MODERATE': 0.12
    }.get(client_severity, 0.1)

    overlap_factor = 1.0 + 0.5 * (1.0 - overlap_ratio)
    val_factor     = 1.0 + 0.3 * (1.0 - val_coverage)
    train_factor   = 1.0 + 0.2 * (1.0 - train_coverage)

    progress     = round_num / max(total_rounds - 1, 1)
    round_factor = 0.3 + 0.7 / (1.0 + np.exp(-10.0 * (progress - 0.3)))

    strength = (base_strength * overlap_factor *
                val_factor    * train_factor   * round_factor)
    return float(np.clip(strength, 0.0, 1.0))


In [ ]:
# Evaluate a client model and return accuracy
def evaluate_user_model(model, data, labels, client_severity,
                        batch_size=32, global_model=None):
    
    pred_gen    = FedPerPredictGenerator(data, batch_size)
    predictions = None
    acc         = 0.0

    try:
        if client_severity in GLOBAL_SEVERITIES and global_model is not None:
            predictions = global_model.predict(pred_gen, verbose=0)
            preds       = predictions
        else:
            predictions = model.predict(pred_gen, verbose=0)
            if isinstance(predictions, dict):
                preds = predictions.get('personal_output',
                        predictions.get('global_output'))
            else:
                preds = predictions

        acc = float(np.mean(
            np.argmax(preds, 1) == np.argmax(labels, 1)))

    except Exception as e:
        print(f"    Eval error: {e}")
        acc = 0.0

    finally:
        del pred_gen
        if predictions is not None:
            del predictions
        gc.collect()

    return acc

In [ ]:
# Decide whether a lightweight architecture trial should run for a user
def should_run_trial(round_num, user_id, client_severities,
                     trial_counts, trial_interval=5, max_trials=9):
    if round_num == 0:
        return False
    if round_num % trial_interval != 0:
        return False
    if client_severities.get(user_id, 'HIGH') not in PERSONAL_SEVERITIES:
        return False
    return trial_counts.get(user_id, 0) < max_trials

# Check for personal head architectures and return the best one
def run_lightweight_trial(user_id, current_model, splits, global_model,
                          input_shape, n_class, domain, severity,
                          num_trials=3, quick_epochs=2, batch_size=32):

    current_backbone = get_backbone_weights(current_model)
    current_global_output = get_global_output_weights(current_model)

    current_best_acc = evaluate_user_model(
        current_model, splits['test_data'],
        splits['test_labels'], severity, batch_size)

    n_samples = len(splits['train_data'])
    if n_samples > 750:
        configs = [
            {'d1': 64,  'd2': 32},
            {'d1': 128, 'd2': 64},
            {'d1': 256, 'd2': 128}]
    else:
        configs = [
            {'d1': 32, 'd2': None},
            {'d1': 64, 'd2': 32},
            {'d1': 96, 'd2': 48}]
    configs = configs[:num_trials]

    # Build DASPR loss once and reuse across all trials
    daspr_loss = AdaptiveDASPRLoss( splits['train_labels'], splits['test_labels'], n_class, severity_override=severity)

    best_trial_model = None
    best_trial_acc   = current_best_acc
    print(f" {user_id}: trial (baseline={current_best_acc:.4f})")

    for idx, cfg in enumerate(configs):
        trial_model = None
        try:
            # Build trial dual-head model
            t_in, t_out  = shared_backbone(input_shape)
            x_flat       = Flatten(name='bb_flatten')(t_out)
            global_out   = Dense(n_class, activation='softmax', name='global_output')(x_flat)
            x_pers       = Dense(cfg['d1'], activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.01), name=f'personal_dense_1_{domain}')(x_flat)
            if cfg['d2']:
                x_pers   = Dense(cfg['d2'], activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.05), name=f'personal_dense_2_{domain}')(x_pers)
            personal_out = Dense(n_class, activation='softmax', name='personal_output')(x_pers)
            trial_model  = Model(
                inputs=t_in,
                outputs={'global_output':   global_out,
                         'personal_output': personal_out})
            
            # Restore backbone and global output from current model
            set_backbone_weights(trial_model, current_backbone)
            if current_global_output:
                set_global_output_weights(trial_model, current_global_output)

            # Freeze backbone for quick training
            for layer in trial_model.layers:
                if any(bb in layer.name for bb in BACKBONE_LAYER_NAMES):
                    layer.trainable = False
            _compile_dual_head_model(trial_model, daspr_loss)

            train_gen = FedPerDualHeadGenerator(
                splits['train_data'], splits['train_labels'],
                batch_size=batch_size, shuffle=True)
            trial_model.fit(train_gen, epochs=quick_epochs, verbose=0)
            del train_gen                        
            gc.collect()

            # Unfreeze for eval
            for layer in trial_model.layers:
                layer.trainable = True
            _compile_dual_head_model(trial_model, daspr_loss)

            acc    = evaluate_user_model(
                trial_model, splits['test_data'],
                splits['test_labels'], severity, batch_size)
            d2_str = f"/{cfg['d2']}" if cfg['d2'] else ""
            print(f"  T{idx+1}: d1={cfg['d1']}{d2_str} → {acc:.4f}"
                  f"{'  ← BEST' if acc > best_trial_acc else ''}")
            
            # Keep this trial model only if it beats the current best
            if acc > best_trial_acc:
                if best_trial_model is not None:
                    safe_model_delete(best_trial_model)  
                    best_trial_model = None              
                best_trial_acc   = acc
                best_trial_model = trial_model
                trial_model      = None                  
            else:
                safe_model_delete(trial_model)
                trial_model = None                     
                gc.collect()

        except Exception as e:
            print(f"      T{idx+1} failed: {e}")
            if trial_model is not None:
                safe_model_delete(trial_model)
                trial_model = None                  
            gc.collect()


    del current_backbone
    if current_global_output is not None:
        del current_global_output
    del daspr_loss
    gc.collect()

    if best_trial_model is not None and best_trial_acc > current_best_acc:
        print(f"    {user_id}: updated (+{best_trial_acc - current_best_acc:.4f})")
        return best_trial_model, True
    else:
        if best_trial_model is not None:
            safe_model_delete(best_trial_model)
            best_trial_model = None                      
        print(f"    {user_id}: no improvement")
        return current_model, False

In [ ]:
# FedProx training
def train_local_fedprox_unified_v2(local_model, global_backbone_weights,
                                    train_data, train_labels,
                                    mu, epochs, batch_size,
                                    val_data=None, val_labels=None,
                                    client_severity='LOW',
                                    round_num=0, total_rounds=50):

    optimizer    = local_model.optimizer
    is_dual_head = (hasattr(local_model, 'output_names') and
                    len(local_model.output_names) > 1)
    
    # Personalized (LOW severity)
    if is_dual_head:
        # Loss functions 
        if hasattr(local_model, '_custom_loss_fns'):
            global_loss_fn   = local_model._custom_loss_fns['global_output']
            personal_loss_fn = local_model._custom_loss_fns['personal_output']
        else:
            global_loss_fn   = local_model.loss.get('global_output',   local_model.loss)
            personal_loss_fn = local_model.loss.get('personal_output', local_model.loss)

        if hasattr(local_model, '_custom_loss_weights'):
            w_global   = local_model._custom_loss_weights['global_output']
            w_personal = local_model._custom_loss_weights['personal_output']
        else:
            w_global, w_personal = 0.3, 0.7 # Personal head will provide more than global

        # Backbone variable IDs 
        backbone_var_ids = set()
        for layer_name in BACKBONE_LAYER_NAMES:
            try:
                layer = local_model.get_layer(layer_name)
                for var in layer.trainable_variables:
                    backbone_var_ids.add(id(var))
            except ValueError:
                continue

        # Backbone proximal references (TF constants)
        backbone_refs = {}
        for layer_name in BACKBONE_LAYER_NAMES:
            try:
                layer = local_model.get_layer(layer_name)
                if layer_name in global_backbone_weights:
                    for var_idx, var in enumerate(layer.trainable_variables):
                        if var_idx < len(global_backbone_weights[layer_name]):
                            backbone_refs[id(var)] = tf.constant(
                                global_backbone_weights[layer_name][var_idx],
                                dtype=tf.float32)
            except ValueError:
                continue

        mu_half = tf.constant(mu / 2.0, dtype=tf.float32)

        history = {'train_accuracy': [], 'val_accuracy': [],
                   'prox_loss': [], 'ce_loss': [], 'total_loss': [],
                   'output_prox_strength': 0.0}

        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)

        for epoch in range(epochs):
            indices    = np.random.permutation(n_samples)
            epoch_ce, epoch_prox, epoch_acc = [], [], []

            for batch_idx in range(n_batches):
                start   = batch_idx * batch_size
                end     = min(start + batch_size, n_samples)
                batch_x = train_data[indices[start:end]]
                batch_y = train_labels[indices[start:end]]

                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)

                    if isinstance(predictions, dict):
                        global_pred   = predictions['global_output']
                        personal_pred = predictions['personal_output']
                    elif isinstance(predictions, (list, tuple)):
                        output_names  = local_model.output_names
                        idx_g         = output_names.index('global_output')
                        idx_p         = output_names.index('personal_output')
                        global_pred   = predictions[idx_g]
                        personal_pred = predictions[idx_p]
                    else:
                        global_pred   = predictions
                        personal_pred = predictions

                    ce_global   = global_loss_fn(batch_y, global_pred)
                    ce_personal = personal_loss_fn(batch_y, personal_pred)
                    ce_loss     = w_global * ce_global + w_personal * ce_personal
                    pred_cls    = tf.argmax(personal_pred, axis=1)

                    if local_model.losses:
                        ce_loss = ce_loss + tf.add_n(local_model.losses)

                    # FedProx proximal term: backbone only    
                    prox_term = tf.constant(0.0, dtype=tf.float32)
                    for var in local_model.trainable_weights:
                        if id(var) in backbone_refs:
                            prox_term += tf.reduce_sum(
                                tf.square(var - backbone_refs[id(var)]))
                    prox_loss  = mu_half * prox_term
                    total_loss = ce_loss + prox_loss

                all_gradients = tape.gradient(
                    total_loss, local_model.trainable_weights)
                
                # Layer-wise gradient clipping 
                # Backbone: tight clipping to preserve shared representations
                # Personal: loose clipping to allow personalization
                backbone_grads_vars = []
                personal_grads_vars = []
                for grad, var in zip(all_gradients, local_model.trainable_weights):
                    if grad is None:
                        continue
                    if id(var) in backbone_var_ids:
                        backbone_grads_vars.append((grad, var))
                    else:
                        personal_grads_vars.append((grad, var))

                if backbone_grads_vars:
                    bb_grads, bb_vars = zip(*backbone_grads_vars)
                    bb_grads_clipped, _ = tf.clip_by_global_norm(
                        list(bb_grads), clip_norm=1.0)
                    optimizer.apply_gradients(zip(bb_grads_clipped, bb_vars))

                if personal_grads_vars:
                    p_grads, p_vars = zip(*personal_grads_vars)
                    p_grads_clipped, _ = tf.clip_by_global_norm(
                        list(p_grads), clip_norm=5.0)
                    optimizer.apply_gradients(zip(p_grads_clipped, p_vars))

                epoch_ce.append(float(ce_loss))
                epoch_prox.append(float(prox_loss))
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))

            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['prox_loss'].append(np.mean(epoch_prox))
            history['total_loss'].append(
                np.mean(epoch_ce) + np.mean(epoch_prox))

            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size)
                history['val_accuracy'].append(val_acc)

        del backbone_refs
        del mu_half
        gc.collect()

        return history
    
    # (MODERATE / HIGH / CRITICAL severity)
    else:
        loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
        all_weights_ref = [tf.constant(w.numpy(), dtype=tf.float32)
                           for w in local_model.trainable_weights]

        mu_half = tf.constant(mu / 2.0, dtype=tf.float32)

        history = {'train_accuracy': [], 'val_accuracy': [],
                   'prox_loss': [], 'ce_loss': [], 'total_loss': [],
                   'output_prox_strength': 0.0}

        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)

        for epoch in range(epochs):
            indices    = np.random.permutation(n_samples)
            epoch_ce, epoch_prox, epoch_acc = [], [], []

            for batch_idx in range(n_batches):
                start   = batch_idx * batch_size
                end     = min(start + batch_size, n_samples)
                batch_x = train_data[indices[start:end]]
                batch_y = train_labels[indices[start:end]]

                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)
                    ce_loss     = loss_fn(batch_y, predictions)

                    prox_term = tf.constant(0.0, dtype=tf.float32)
                    for local_w, global_w in zip(
                            local_model.trainable_weights, all_weights_ref):
                        prox_term += tf.reduce_sum(
                            tf.square(local_w - global_w))
                    prox_loss  = mu_half * prox_term
                    total_loss = ce_loss + prox_loss

                gradients = tape.gradient(
                    total_loss, local_model.trainable_weights)
                optimizer.apply_gradients(
                    zip(gradients, local_model.trainable_weights))

                epoch_ce.append(float(ce_loss))
                epoch_prox.append(float(prox_loss))
                pred_cls = tf.argmax(predictions, axis=1)
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))

            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['prox_loss'].append(np.mean(epoch_prox))
            history['total_loss'].append(
                np.mean(epoch_ce) + np.mean(epoch_prox))

            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size)
                history['val_accuracy'].append(val_acc)

        del all_weights_ref
        del mu_half
        del loss_fn
        gc.collect()

        return history

In [ ]:
# FedAvg Functions
# Compute normalized aggregation weights for FedAvg backbone averaging
def compute_quality_weights(user_ids, per_user_splits, severity_metrics):

    raw_weights = {}
    for uid in user_ids:
        data_size     = len(per_user_splits[uid]['train_data'])
        overlap_ratio = severity_metrics[uid]['overlap_ratio']
        raw_weights[uid] = data_size * (0.3 + 0.7 * overlap_ratio)

    total      = sum(raw_weights.values())
    normalized = {uid: w / total for uid, w in raw_weights.items()}
    return normalized

# Compute a weighted average of backbone layer weights across clients
def federated_avg_backbone(client_models, client_weights, verbose=False):
    all_backbones = {}

    for uid, model in client_models.items():
        weight   = client_weights.get(uid, 1.0 / len(client_models))
        backbone = get_backbone_weights(model)
        for layer_name, layer_weights in backbone.items():
            if layer_name not in all_backbones:
                all_backbones[layer_name] = []
            all_backbones[layer_name].append((layer_weights, weight, uid))

    averaged_backbone = {}

    for layer_name in list(all_backbones.keys()):   
        contributions = all_backbones[layer_name]
        if not contributions:
            continue

        # Filter to clients whose weight shapes match the first client
        ref_shapes = [w.shape for w in contributions[0][0]]
        compatible = [
            (ws, wt, uid) for ws, wt, uid in contributions
            if [w.shape for w in ws] == ref_shapes
        ]

        if not compatible:
            if verbose:
                print(f" {layer_name}: no compatible weights")
            del all_backbones[layer_name]            
            continue

        # Weighted average across compatible clients
        total_weight = sum(wt for _, wt, _ in compatible)
        n_arrays     = len(compatible[0][0])

        averaged_layer = []
        for arr_idx in range(n_arrays):
            weighted_sum = np.zeros_like(compatible[0][0][arr_idx])
            for layer_ws, user_wt, _ in compatible:
                weighted_sum += layer_ws[arr_idx] * (user_wt / total_weight)
            averaged_layer.append(weighted_sum)

        averaged_backbone[layer_name] = averaged_layer

        del compatible
        del all_backbones[layer_name]
        gc.collect()

        if verbose:
            print(f"  {layer_name}: averaged")

    del all_backbones
    gc.collect()

    return averaged_backbone

# Compute a weighted average of the global_output layer weights.
def federated_avg_global_output(client_models, client_weights, verbose=False):
    contributions = []
    weights_list  = []
    user_ids      = []

    # Collect global_output weights from all clients that have the layer
    for uid, model in client_models.items():
        ws = get_global_output_weights(model)
        if ws is not None:
            contributions.append(ws)
            weights_list.append(
                client_weights.get(uid, 1.0 / len(client_models)))
            user_ids.append(uid)

    if not contributions:
        if verbose:
            print(f" global_output: no contributions")
        return None

    total_w = sum(weights_list)
    if total_w == 0:
        return None

    ref_shapes = [w.shape for w in contributions[0]]
    compatible = [
        (ws, wt, uid)
        for ws, wt, uid in zip(contributions, weights_list, user_ids)
        if [w.shape for w in ws] == ref_shapes
    ]

    if not compatible:
        if verbose:
            print(f"   global_output: no compatible weights")
        del contributions, weights_list, user_ids  
        gc.collect()
        return None
    
    # Weighted average across compatible clients
    total_w = sum(wt for _, wt, _ in compatible)
    avg = []
    for i in range(len(compatible[0][0])):
        weighted_sum = np.zeros_like(compatible[0][0][i])
        for ws, uw, _ in compatible:
            weighted_sum += ws[i] * (uw / total_w)
        avg.append(weighted_sum)

    del compatible
    del contributions
    del weights_list
    del user_ids
    gc.collect()

    if verbose:
        print(f" global_output: averaged from {len(avg)} arrays")

    return avg

In [ ]:
# EWC IMPLEMENTATION FOR NEW CLIENT ADAPTATION
# Pre-computes and caches TensorFlow constants for EWC penalty computation
class EWCCache:
    def __init__(self):
        self._cache           = {}
        self._lambda          = None
        self._layer_to_weights = None

    def build(self, reference_weights_numpy, fisher_numpy,
              layer_to_weights, lambda_ewc):
        self._cache.clear()
        self._lambda           = tf.constant(lambda_ewc, dtype=tf.float32)
        self._layer_to_weights = layer_to_weights

        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                if (fisher_numpy[wi] is not None and
                        reference_weights_numpy[wi] is not None):
                    self._cache[wi] = (
                        tf.constant(reference_weights_numpy[wi],
                                    dtype=tf.float32),
                        tf.constant(fisher_numpy[wi],
                                    dtype=tf.float32)
                    )
        print(f"  EWCCache built: {len(self._cache)} weight arrays cached")
        return self

    def refresh(self, reference_weights_numpy, fisher_numpy, lambda_ewc):
        self._cache.clear()
        gc.collect()
        return self.build(reference_weights_numpy, fisher_numpy,
                          self._layer_to_weights, lambda_ewc)

    def is_ready(self):
        return len(self._cache) > 0

    def compute_loss(self, model):
        if not self.is_ready():
            return tf.constant(0.0, dtype=tf.float32)

        ewc_loss = tf.constant(0.0, dtype=tf.float32)

        trainable_map = {}
        global_idx = 0
        t_idx      = 0
        for layer in model.layers:
            for _ in layer.get_weights():
                if layer.trainable:
                    trainable_map[global_idx] = model.trainable_weights[t_idx]
                    t_idx += 1
                global_idx += 1

        for wi, (ref_tf, fisher_tf) in self._cache.items():
            if wi not in trainable_map:
                continue
            current_w = trainable_map[wi]
            if current_w.shape == ref_tf.shape == fisher_tf.shape:
                diff     = current_w - ref_tf
                penalty  = tf.reduce_sum(fisher_tf * tf.square(diff))
                ewc_loss = ewc_loss + penalty

        del trainable_map   
        return self._lambda * ewc_loss


# Compute the EWC penalty term for a model
def compute_ewc_loss_numpy_refs(model, reference_weights_numpy,
                                 fisher_numpy, layer_to_weights,
                                 lambda_ewc, ewc_cache=None):
    
    if ewc_cache is not None and ewc_cache.is_ready():
        return ewc_cache.compute_loss(model)

    # Fallback path
    ewc_loss   = tf.constant(0.0, dtype=tf.float32)
    trainable_map = {}
    global_idx    = 0
    t_idx         = 0
    for layer in model.layers:
        for _ in layer.get_weights():
            if layer.trainable:
                trainable_map[global_idx] = model.trainable_weights[t_idx]
                t_idx += 1
            global_idx += 1

    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if (wi not in trainable_map or
                    fisher_numpy[wi] is None or
                    reference_weights_numpy[wi] is None):
                continue
            current_w   = trainable_map[wi]
            reference_w = tf.constant(reference_weights_numpy[wi],
                                      dtype=tf.float32)
            fisher      = tf.constant(fisher_numpy[wi], dtype=tf.float32)
            if current_w.shape == reference_w.shape == fisher.shape:
                diff     = current_w - reference_w
                penalty  = tf.reduce_sum(fisher * tf.square(diff))
                ewc_loss = ewc_loss + penalty
            del reference_w, fisher  
    del trainable_map                
    return lambda_ewc * ewc_loss


# training for a new G2 client using EWC regularisation
def train_new_client_with_ewc(
        model, train_data, train_labels,
        reference_weights_numpy, fisher_numpy,
        layer_to_weights, lambda_ewc,
        epochs, batch_size,
        val_data=None, val_labels=None,
        patience=5, verbose=1,
        ewc_cache=None):

    optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001)

    history = {
        'train_loss'    : [], 'train_accuracy' : [],
        'train_ce_loss' : [], 'train_ewc_loss' : [],
        'val_loss'      : [], 'val_accuracy'   : []
    }

    best_val_loss    = float('inf')
    patience_counter = 0

    use_ewc = (len(layer_to_weights) > 0 and
               lambda_ewc > 0 and
               any(fisher_numpy[wi] is not None
                   for wi_list in layer_to_weights.values()
                   for wi in wi_list))

    if verbose >= 1:
        print(f"  NEW CLIENT TRAINING WITH EWC")
        print(f"  EWC enabled : {use_ewc}")
        print(f"  Lambda EWC  : {lambda_ewc}")
        print(f"  Epochs      : {epochs}")
        print(f"  Train size  : {len(train_data)}")

    early_stop = False

    for epoch in range(epochs):
        perm            = np.random.permutation(len(train_data))
        shuffled_data   = train_data[perm]   
        shuffled_labels = train_labels[perm]  

        epoch_ce  = 0.0
        epoch_ewc = 0.0
        epoch_acc = 0.0
        n_batches = 0

        for start in range(0, len(shuffled_data), batch_size):
            end     = min(start + batch_size, len(shuffled_data))
            batch_x = tf.constant(shuffled_data[start:end],   dtype=tf.float32)
            batch_y = tf.constant(shuffled_labels[start:end], dtype=tf.float32)

            with tf.GradientTape() as tape:
                preds   = model(batch_x, training=True)
                ce_loss = tf.reduce_mean(
                    tf.keras.losses.categorical_crossentropy(batch_y, preds))

                if use_ewc:
                    ewc_penalty = compute_ewc_loss_numpy_refs(
                        model, reference_weights_numpy, fisher_numpy,
                        layer_to_weights, lambda_ewc, ewc_cache=ewc_cache)
                    total_loss = ce_loss + ewc_penalty
                else:
                    ewc_penalty = tf.constant(0.0)
                    total_loss  = ce_loss

            # Per-gradient norm clipping to stabilise warm-up training
            grads = tape.gradient(total_loss, model.trainable_weights)
            grads = [tf.clip_by_norm(g, 1.0) if g is not None else g
                     for g in grads]
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

            epoch_ce  += ce_loss.numpy()
            epoch_ewc += ewc_penalty.numpy()
            epoch_acc += tf.reduce_mean(
                tf.keras.metrics.categorical_accuracy(batch_y, preds)
            ).numpy()
            n_batches += 1

            del tape, preds, ce_loss, ewc_penalty, total_loss, grads
            del batch_x, batch_y  

        del shuffled_data, shuffled_labels

        # Normalise epoch metrics by number of batches
        epoch_ce  /= n_batches
        epoch_ewc /= n_batches
        epoch_acc /= n_batches

        history['train_ce_loss'].append(epoch_ce)
        history['train_ewc_loss'].append(epoch_ewc)
        history['train_loss'].append(epoch_ce + epoch_ewc)
        history['train_accuracy'].append(epoch_acc)

        if verbose >= 2:
            print(f"  Epoch {epoch+1:>3}/{epochs} | "
                  f"CE={epoch_ce:.4f}  EWC={epoch_ewc:.4f}  "
                  f"Total={epoch_ce+epoch_ewc:.4f}  Acc={epoch_acc:.4f}")

        if val_data is not None and val_labels is not None:
            val_preds = model.predict(
                val_data, verbose=0, batch_size=batch_size)
            val_loss  = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(
                    val_labels.astype(np.float32), val_preds)
            ).numpy()
            val_acc   = np.mean(
                np.argmax(val_preds, axis=1) == np.argmax(val_labels, axis=1))
            del val_preds  
            gc.collect()

            history['val_loss'].append(val_loss)
            history['val_accuracy'].append(val_acc)

            if verbose >= 2:
                print(f"   Val  | "
                      f"loss={val_loss:.4f}  acc={val_acc:.4f}")

            if val_loss < best_val_loss:
                best_val_loss    = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch+1}")
                    early_stop = True
        gc.collect()
        if early_stop:
            break
    return history


# Automatically recommend ewc_sample_ratio and Fisher batch_size
def auto_select_ewc_params(group1_per_user,
                            target_fisher_time_seconds=30,
                            batch_size=32,
                            seconds_per_batch=0.15):

    total_samples = sum(
        len(s['train_data']) for s in group1_per_user.values())
    n_users = len(group1_per_user)

    # Minimum ratio to cover all active classes adequately
    avg_active_classes = np.mean([
        np.sum(np.sum(s['train_labels'], axis=0) > 0)
        for s in group1_per_user.values()])
    min_samples_needed = int(avg_active_classes * 10 * n_users)
    min_ratio          = max(0.05, min(min_samples_needed / total_samples, 0.15))

    max_batches = target_fisher_time_seconds / seconds_per_batch
    max_samples = int(max_batches * batch_size)
    time_ratio  = min(max_samples / total_samples, 1.0)

    final_ratio    = float(np.clip(max(min_ratio, time_ratio), 0.05, 0.50))
    final_ratio    = round(final_ratio, 2)
    sampled_n      = int(total_samples * final_ratio)
    rec_batch_size = min(128, max(32, sampled_n // 20))
    rec_batch_size = int(2 ** round(np.log2(rec_batch_size)))

    print(f"\n  Recommended ewc_sample_ratio : {final_ratio}")
    print(f"  Recommended Fisher batch_size: {rec_batch_size}")
    return final_ratio, rec_batch_size


# Local Fisher Computation (runs on each client)
def compute_local_fisher_numpy(model, train_data, train_labels,
                                layer_to_weights, batch_size=32,
                                sample_ratio=0.3, user_id=None):

    n_samples = max(1, int(len(train_data) * sample_ratio))
    idx = np.random.choice(len(train_data), n_samples, replace=False)
    sampled_data   = train_data[idx]
    sampled_labels = train_labels[idx]
    
    if user_id:
        print(f"    {user_id}: computing local Fisher "
              f"({n_samples}/{len(train_data)} samples)")
    
    # Initialize local Fisher (same shape as global weights)
    all_weights  = model.get_weights()
    total_w      = len(all_weights)
    local_fisher = [None] * total_w
    
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            local_fisher[wi] = np.zeros_like(
                all_weights[wi], dtype=np.float32)
    del all_weights
    
    # Build trainable weight index map
    trainable_to_global = {}
    global_idx = 0
    t_idx      = 0
    for layer in model.layers:
        for _ in layer.get_weights():
            if layer.trainable:
                trainable_to_global[t_idx] = global_idx
                t_idx += 1
            global_idx += 1
    global_to_trainable = {v: k for k, v in trainable_to_global.items()}
    del trainable_to_global
    
    # Accumulate squared gradients
    n_batches = max(1, (len(sampled_data) + batch_size - 1) // batch_size)
    
    for batch_idx in range(n_batches):
        start   = batch_idx * batch_size
        end     = min(start + batch_size, len(sampled_data))
        batch_x = tf.constant(sampled_data[start:end], dtype=tf.float32)
        batch_y = tf.constant(sampled_labels[start:end], dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            preds = model(batch_x, training=False)
            loss  = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(batch_y, preds))
        
        grads = tape.gradient(loss, model.trainable_weights)
        
        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                t_i = global_to_trainable.get(wi)
                if t_i is not None and grads[t_i] is not None:
                    local_fisher[wi] += grads[t_i].numpy() ** 2
        del tape, preds, loss, grads, batch_x, batch_y
    
    # Normalize by number of batches
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if local_fisher[wi] is not None:
                local_fisher[wi] /= float(n_batches)
    
    del global_to_trainable
    del sampled_data, sampled_labels
    gc.collect()
    return local_fisher, n_samples


# Server-Side Federated Aggregation
def aggregate_fisher_federated(client_fishers, layer_to_weights,
                                 aggregation='weighted',
                                 severity_metrics=None,
                                 verbose=True):
    if not client_fishers:
        return None
    
    # Compute aggregation weights 
    if aggregation == 'weighted':
        # Weight by number of samples used for Fisher computation
        total_samples = sum(n for _, n in client_fishers.values())
        agg_weights = {
            uid: n / total_samples
            for uid, (_, n) in client_fishers.items()
        }
        
    elif aggregation == 'quality_weighted':
        # Weight by samples × overlap_ratio (same as FedAvg backbone)
        raw_weights = {}
        for uid, (_, n_samples) in client_fishers.items():
            overlap = (severity_metrics[uid]['overlap_ratio']
                       if severity_metrics and uid in severity_metrics
                       else 1.0)
            raw_weights[uid] = n_samples * (0.3 + 0.7 * overlap)
        total = sum(raw_weights.values())
        agg_weights = {uid: w / total for uid, w in raw_weights.items()}
        
    else:  # equal
        n_clients   = len(client_fishers)
        agg_weights = {uid: 1.0 / n_clients for uid in client_fishers}
    
    if verbose:
        print(f"\n    Federated Fisher Aggregation ({aggregation}):")
        for uid, w in sorted(agg_weights.items()):
            _, n = client_fishers[uid]
            print(f"      {uid}: weight={w:.4f} (n_samples={n})")
    
    # Get reference structure 
    ref_fisher = next(iter(client_fishers.values()))[0]
    total_w    = len(ref_fisher)
    
    aggregated_fisher = [None] * total_w
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            aggregated_fisher[wi] = np.zeros_like(
                ref_fisher[wi], dtype=np.float32)
    
    #  Weighted sum 
    for uid, (local_fisher, _) in client_fishers.items():
        w = agg_weights[uid]
        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                if (local_fisher[wi] is not None and
                        aggregated_fisher[wi] is not None):
                    # Shape safety check
                    if local_fisher[wi].shape == aggregated_fisher[wi].shape:
                        aggregated_fisher[wi] += w * local_fisher[wi]
    
    if verbose:
        n_aggregated = sum(
            1 for layer_idx, wi_list in layer_to_weights.items()
            for wi in wi_list
            if aggregated_fisher[wi] is not None)
        print(f"    ✓ Aggregated {n_aggregated} weight arrays "
              f"from {len(client_fishers)} clients")   
    return aggregated_fisher


# Federated replacement for create_fisher_numpy
def create_fisher_federated(model, per_user_splits, shared_layer_indices,
                              batch_size=32, sample_ratio=0.3,
                              aggregation='quality_weighted',
                              severity_metrics=None,
                              verbose=True):

    print(f"\n  Computing Federated Fisher Information...")
    print(f"  Clients: {sorted(per_user_splits.keys())}")
    print(f"  Aggregation: {aggregation}")
    
    # Build layer_to_weights (server-side, uses model structure) 
    layer_to_weights = {}
    weight_idx = 0
    for layer_idx, layer in enumerate(model.layers):
        if layer.trainable and layer.get_weights():
            num_w = len(layer.get_weights())
            if layer_idx in shared_layer_indices:
                layer_to_weights[layer_idx] = list(
                    range(weight_idx, weight_idx + num_w))
            weight_idx += num_w
    
    print(f"  Shared layers for Fisher: "
          f"{len(layer_to_weights)} "
          f"(indices: {list(layer_to_weights.keys())})")
    
    # Each client computes LOCAL Fisher 
    client_fishers = {}
    
    for user_id in sorted(per_user_splits.keys()):
        splits = per_user_splits[user_id]
        try:
            local_fisher, n_used = compute_local_fisher_numpy(
                model=model,
                train_data=splits['train_data'],
                train_labels=splits['train_labels'],
                layer_to_weights=layer_to_weights,
                batch_size=batch_size,
                sample_ratio=sample_ratio,
                user_id=user_id)
            
            client_fishers[user_id] = (local_fisher, n_used)
            
        except Exception as e:
            print(f"    {user_id}: Fisher computation failed — {e}")
            import traceback; traceback.print_exc()
        finally:
            gc.collect()
    print(f"\n  Received Fisher from "
          f"{len(client_fishers)}/{len(per_user_splits)} clients")
    
    # Server aggregates 
    aggregated_fisher = aggregate_fisher_federated(
        client_fishers=client_fishers,
        layer_to_weights=layer_to_weights,
        aggregation=aggregation,
        severity_metrics=severity_metrics,
        verbose=verbose)
    
    for uid in list(client_fishers.keys()):
        del client_fishers[uid]
    del client_fishers
    gc.collect()
    
    print(f"\n  Federated Fisher computed for {len(layer_to_weights)} layers")
    return aggregated_fisher, layer_to_weights


# Federated Fisher Refresh with EMA smoothing
def refresh_fisher_federated(model, per_user_splits, shared_layer_indices,
                               old_fisher_numpy, layer_to_weights,
                               batch_size=32, sample_ratio=0.3,
                               ema_alpha=0.3,
                               aggregation='quality_weighted',
                               severity_metrics=None):

    print(f"\n  Federated Fisher Refresh (EMA α={ema_alpha})")
    
    new_fisher, _ = create_fisher_federated(
        model=model,
        per_user_splits=per_user_splits,
        shared_layer_indices=shared_layer_indices,
        batch_size=batch_size,
        sample_ratio=sample_ratio,
        aggregation=aggregation,
        severity_metrics=severity_metrics,
        verbose=False)
    
    # EMA update (reuse your existing function)
    updated_fisher = update_fisher_numpy_ema(
        old_fisher_numpy=old_fisher_numpy,
        new_fisher_numpy=new_fisher,
        layer_to_weights=layer_to_weights,
        alpha=ema_alpha)
    
    del new_fisher
    gc.collect()
    print(f"  ✓ Federated Fisher refreshed (EMA)")
    return updated_fisher


# Apply exponential moving average to update Fisher matrices in-place
def update_fisher_numpy_ema(old_fisher_numpy, new_fisher_numpy,
                             layer_to_weights, alpha=0.3):
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if (old_fisher_numpy[wi] is not None and
                    new_fisher_numpy[wi] is not None):
                old_fisher_numpy[wi] *= (1 - alpha)
                old_fisher_numpy[wi] += alpha * new_fisher_numpy[wi]
                new_fisher_numpy[wi]  = None 
    gc.collect()
    return old_fisher_numpy

In [ ]:
# Run per round of local EWC training for a G2 (new) client
def train_local_fedprox_with_ewc(
        local_model, 
        global_backbone_weights,
        train_data, 
        train_labels,
        mu,                       
        epochs, 
        batch_size,
        val_data=None, 
        val_labels=None,
        client_severity='LOW',
        reference_weights_numpy=None,
        fisher_numpy=None,
        layer_to_weights=None,
        lambda_ewc=0.4,
        round_num=0, 
        total_rounds=50,
        ewc_cache=None):

    
    optimizer = local_model.optimizer
    is_dual_head = (hasattr(local_model, 'output_names') and
                    len(local_model.output_names) > 1)
    
    # EWC ACTIVE CHECK
    use_ewc = (layer_to_weights is not None and
               fisher_numpy is not None and
               reference_weights_numpy is not None and
               lambda_ewc > 0 and
               any(fisher_numpy[wi] is not None
                   for wi_list in layer_to_weights.values()
                   for wi in wi_list))
    
    if not use_ewc:
        print(f" G2 Training: EWC disabled (missing params)")
    else:
        print(f" G2 Training: EWC (λ={lambda_ewc:.3f}) [{client_severity}]")
    
    # LOW SEVERITY Clients
    if is_dual_head:
        
        # Loss functions 
        if hasattr(local_model, '_custom_loss_fns'):
            global_loss_fn = local_model._custom_loss_fns['global_output']
            personal_loss_fn = local_model._custom_loss_fns['personal_output']
        else:
            global_loss_fn = local_model.loss.get('global_output', local_model.loss)
            personal_loss_fn = local_model.loss.get('personal_output', local_model.loss)
        
        if hasattr(local_model, '_custom_loss_weights'):
            w_global = local_model._custom_loss_weights['global_output']
            w_personal = local_model._custom_loss_weights['personal_output']
        else:
            w_global, w_personal = 0.3, 0.7
        
        # Identify backbone vs personal variables 
        backbone_var_ids = set()
        for layer_name in BACKBONE_LAYER_NAMES:
            try:
                layer = local_model.get_layer(layer_name)
                for var in layer.trainable_variables:
                    backbone_var_ids.add(id(var))
            except ValueError:
                continue
        
        history = {
            'train_accuracy': [], 'val_accuracy': [],
            'ce_loss': [], 'ewc_loss': [], 'total_loss': []}
        
        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)
        
        # Training loop 
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            epoch_ce, epoch_ewc, epoch_acc = [], [], []
            
            for batch_idx in range(n_batches):
                start = batch_idx * batch_size
                end = min(start + batch_size, n_samples)
                
                batch_x = tf.constant(train_data[indices[start:end]], dtype=tf.float32)
                batch_y = tf.constant(train_labels[indices[start:end]], dtype=tf.float32)
                
                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)
                    
                    # Extract predictions
                    if isinstance(predictions, dict):
                        global_pred = predictions['global_output']
                        personal_pred = predictions['personal_output']
                    elif isinstance(predictions, (list, tuple)):
                        output_names = local_model.output_names
                        idx_g = output_names.index('global_output')
                        idx_p = output_names.index('personal_output')
                        global_pred = predictions[idx_g]
                        personal_pred = predictions[idx_p]
                    else:
                        global_pred = predictions
                        personal_pred = predictions
                    
                    # DASPR loss
                    ce_global = global_loss_fn(batch_y, global_pred)
                    ce_personal = personal_loss_fn(batch_y, personal_pred)
                    ce_loss = w_global * ce_global + w_personal * ce_personal
                    pred_cls = tf.argmax(personal_pred, axis=1)
                    
                    if local_model.losses:
                        ce_loss = ce_loss + tf.add_n(local_model.losses)
                    
                    # EWC penalty (on shared backbone layers)
                    if use_ewc:
                        ewc_penalty = compute_ewc_loss_numpy_refs(
                            local_model,
                            reference_weights_numpy,
                            fisher_numpy,
                            layer_to_weights,
                            lambda_ewc,
                            ewc_cache=ewc_cache)
                    else:
                        ewc_penalty = tf.constant(0.0, dtype=tf.float32)
                    
                    total_loss = ce_loss + ewc_penalty
                
                # Layer-wise gradient clipping (DASPR style)
                all_gradients = tape.gradient(total_loss, local_model.trainable_weights)
                
                backbone_grads_vars = []
                personal_grads_vars = []
                
                for grad, var in zip(all_gradients, local_model.trainable_weights):
                    if grad is None:
                        continue
                    if id(var) in backbone_var_ids:
                        backbone_grads_vars.append((grad, var))
                    else:
                        personal_grads_vars.append((grad, var))
                
                # Backbone: TIGHT clipping (regularized by EWC)
                if backbone_grads_vars:
                    bb_grads, bb_vars = zip(*backbone_grads_vars)
                    bb_grads_clipped, _ = tf.clip_by_global_norm(
                        list(bb_grads), clip_norm=1.0)
                    optimizer.apply_gradients(zip(bb_grads_clipped, bb_vars))
                
                # Personal: LOOSE clipping (not regularized)
                if personal_grads_vars:
                    p_grads, p_vars = zip(*personal_grads_vars)
                    p_grads_clipped, _ = tf.clip_by_global_norm(
                        list(p_grads), clip_norm=5.0)
                    optimizer.apply_gradients(zip(p_grads_clipped, p_vars))
                
                # Metrics
                epoch_ce.append(float(ce_loss))
                epoch_ewc.append(float(ewc_penalty))
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))
                
                del tape, predictions, ce_loss, ewc_penalty, total_loss
                del all_gradients, batch_x, batch_y
            
            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['ewc_loss'].append(np.mean(epoch_ewc))
            history['total_loss'].append(np.mean(epoch_ce) + np.mean(epoch_ewc))
            
            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size)
                history['val_accuracy'].append(val_acc)
            
            gc.collect()
        
        return history
    

    # HIGH/MOD/CRIT SEVERITY 
    else:
        loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
        
        history = {
            'train_accuracy': [], 'val_accuracy': [],
            'ce_loss': [], 'ewc_loss': [], 'total_loss': []}
        
        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)
        
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            epoch_ce, epoch_ewc, epoch_acc = [], [], []
            
            for batch_idx in range(n_batches):
                start = batch_idx * batch_size
                end = min(start + batch_size, n_samples)
                
                batch_x = tf.constant(train_data[indices[start:end]], dtype=tf.float32)
                batch_y = tf.constant(train_labels[indices[start:end]], dtype=tf.float32)
                
                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)
                    ce_loss = loss_fn(batch_y, predictions)
                    
                    # EWC penalty (on shared backbone layers)
                    if use_ewc:
                        ewc_penalty = compute_ewc_loss_numpy_refs(
                            local_model,
                            reference_weights_numpy,
                            fisher_numpy,
                            layer_to_weights,
                            lambda_ewc,
                            ewc_cache=ewc_cache)
                    else:
                        ewc_penalty = tf.constant(0.0, dtype=tf.float32)
                    
                    total_loss = ce_loss + ewc_penalty
                
                # Standard gradient update
                gradients = tape.gradient(total_loss, local_model.trainable_weights)
                optimizer.apply_gradients(zip(gradients, local_model.trainable_weights))
                
                # Metrics
                epoch_ce.append(float(ce_loss))
                epoch_ewc.append(float(ewc_penalty))
                pred_cls = tf.argmax(predictions, axis=1)
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))
                
                del tape, predictions, ce_loss, ewc_penalty, total_loss
                del gradients, batch_x, batch_y
            
            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['ewc_loss'].append(np.mean(epoch_ewc))
            history['total_loss'].append(np.mean(epoch_ce) + np.mean(epoch_ewc))
            
            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size)
                history['val_accuracy'].append(val_acc)
            
            gc.collect()
        
        del loss_fn
        gc.collect()
        
        return history

In [ ]:
import psutil
import time
from collections import defaultdict
import gc

class ResourceMonitor:
    
    def __init__(self, enable_gpu=False):
        self.process = psutil.Process()
        self.enable_gpu = enable_gpu
        
        # Round-level metrics (lightweight)
        self.round_metrics = defaultdict(dict)
        
        # Model metrics (measured once per architecture)
        self.model_cache = {}
        self._measured_archs = set()
        
        # Baseline
        self.baseline_memory = self.process.memory_info().rss / (1024 * 1024)
    
    def mark_round_start(self, round_num):
        """Mark start of round (fast: ~0.001s)"""
        self.round_metrics[round_num]['start_time'] = time.time()
        self.round_metrics[round_num]['start_memory'] = self.process.memory_info().rss / (1024 * 1024)
    
    def mark_round_end(self, round_num):
        """Mark end of round and calculate metrics (fast: ~0.001s)"""
        end_time = time.time()
        end_memory = self.process.memory_info().rss / (1024 * 1024)
        
        self.round_metrics[round_num].update({
            'end_time': end_time,
            'end_memory': end_memory,
            'duration': end_time - self.round_metrics[round_num]['start_time'],
            'memory_delta': end_memory - self.round_metrics[round_num]['start_memory'],
            'peak_memory': max(self.round_metrics[round_num]['start_memory'], end_memory)
        })
    
    def measure_model_resources(self, model, model_name):

        # Create architecture fingerprint
        arch_id = f"{len(model.layers)}_{model.count_params()}"
        
        # Return cached if already measured
        if arch_id in self._measured_archs:
            return self.model_cache.get(model_name, {})
        
        # Measure (only once per unique architecture)
        param_count = model.count_params()
        trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
        
        metrics = {
            'model_name': model_name,
            'total_params': param_count,
            'trainable_params': trainable_params,
            'non_trainable_params': param_count - trainable_params,
            'memory_mb': (param_count * 4) / (1024 * 1024),
            'total_layers': len(model.layers)}
        
        self.model_cache[model_name] = metrics
        self._measured_archs.add(arch_id)
        return metrics
    
    def get_summary(self):
        import numpy as np
        
        if not self.round_metrics:
            return {'error': 'No data collected'}
        
        durations = [m['duration'] for m in self.round_metrics.values() if 'duration' in m]
        memory_deltas = [m['memory_delta'] for m in self.round_metrics.values() if 'memory_delta' in m]
        peak_memories = [m['peak_memory'] for m in self.round_metrics.values() if 'peak_memory' in m]
        
        return {
            'baseline_memory_mb': self.baseline_memory,
            'total_rounds': len(self.round_metrics),
            'total_time': np.sum(durations) if durations else 0,
            'avg_time_per_round': np.mean(durations) if durations else 0,
            'avg_memory_delta': np.mean(memory_deltas) if memory_deltas else 0,
            'max_peak_memory': np.max(peak_memories) if peak_memories else 0,
            'total_models': len(self.model_cache),
            'unique_architectures': len(self._measured_archs)}
    
    def print_summary(self):
        summary = self.get_summary()
        if 'error' in summary:
            print(f" {summary['error']}")
            return
        
        print(f" RESOURCE SUMMARY")
        print(f"  Total Time: {summary['total_time']:.2f}s ({summary['total_time']/60:.2f} min)")
        print(f"   Avg/Round:  {summary['avg_time_per_round']:.2f}s")
        print(f"   Total Rounds: {summary['total_rounds']}")
        print(f"\n Memory:")
        print(f"   Baseline:  {summary['baseline_memory_mb']:.2f} MB")
        print(f"   Peak:   {summary['max_peak_memory']:.2f} MB")
        print(f"   Avg Delta:  {summary['avg_memory_delta']:.2f} MB/round")
        print(f"\n Models:")
        print(f"  Total:    {summary['total_models']}")
        print(f"   Unique Archs:   {summary['unique_architectures']}")

# Initialize global resource monitor (GPU disabled by default for speed)
resource_monitor = ResourceMonitor(enable_gpu=False)

In [ ]:
# Comprehensive evaluation with precision, recall, F1
def evaluate_user_model_comprehensive(model, test_data, test_labels, severity, 
                                     batch_size=32, global_model=None):
    try:
        # Get predictions
        if severity in PERSONAL_SEVERITIES:
            predictions = model.predict(test_data, batch_size=batch_size, verbose=0)
            if isinstance(predictions, dict):
                y_pred_proba = predictions['personal_output']
            elif isinstance(predictions, (list, tuple)):
                y_pred_proba = predictions[1] if len(predictions) > 1 else predictions[0]
            else:
                y_pred_proba = predictions
        else:
            if global_model is not None:
                y_pred_proba = global_model.predict(test_data, batch_size=batch_size, verbose=0)
            else:
                y_pred_proba = model.predict(test_data, batch_size=batch_size, verbose=0)
        
        if not isinstance(y_pred_proba, np.ndarray):
            raise ValueError(f"Predictions must be numpy array, got {type(y_pred_proba)}")
        
        if len(y_pred_proba.shape) != 2:
            raise ValueError(f"Expected 2D predictions (samples, classes), got shape {y_pred_proba.shape}")
        
        # Get predicted classes
        y_pred = np.argmax(y_pred_proba, axis=1)
        

        if not isinstance(test_labels, np.ndarray):
            test_labels = np.array(test_labels)
            
        if len(test_labels.shape) == 1:
            y_true = test_labels.astype(int)
        else:
            y_true = np.argmax(test_labels, axis=1)
        
        min_len = min(len(y_pred), len(y_true))
        if len(y_pred) != len(y_true):
            print(f"   Warning: prediction length ({len(y_pred)}) != label length ({len(y_true)})")
            y_pred = y_pred[:min_len]
            y_true = y_true[:min_len]
        
        accuracy = np.mean(y_pred == y_true)
        
        # Macro averages
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)
        
        # Weighted averages
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)
        
        return {
            'accuracy': float(accuracy),
            'precision_macro': float(precision_macro),
            'recall_macro': float(recall_macro),
            'f1_macro': float(f1_macro),
            'precision_weighted': float(precision_weighted),
            'recall_weighted': float(recall_weighted),
            'f1_weighted': float(f1_weighted)}
    
    except Exception as e:
        print(f" Evaluation error for severity {severity}: {e}")
        print(f" Model type: {type(model)}")
        print(f" Test data shape: {test_data.shape}")
        print(f" Test labels shape: {test_labels.shape}")
        if 'predictions' in locals():
            print(f" Raw predictions type: {type(predictions)}")
            if isinstance(predictions, dict):
                print(f" Dict keys: {predictions.keys()}")
                for k, v in predictions.items():
                    print(f"   {k} shape: {v.shape if isinstance(v, np.ndarray) else type(v)}")
        if 'y_pred_proba' in locals():
            print(f"  y_pred_proba type: {type(y_pred_proba)}")
            if isinstance(y_pred_proba, np.ndarray):
                print(f"   y_pred_proba shape: {y_pred_proba.shape}")
        
        import traceback
        traceback.print_exc()
        
        # Return default metrics on error
        return {
            'accuracy': 0.0,
            'precision_macro': 0.0,
            'recall_macro': 0.0,
            'f1_macro': 0.0,
            'precision_weighted': 0.0,
            'recall_weighted': 0.0,
            'f1_weighted': 0.0}

In [ ]:
# Full two-phase federated training with personalisation and EWC
def train_fedprox_personalized_with_new_clients(group1_per_user, group1_splits,
                                                group2_per_user, group2_splits,
                                                input_shape, n_class,
                                                case='case1',  
                                                num_rounds_initial=5,
                                                num_rounds_combined=15,
                                                local_epochs=2,
                                                batch_size=32,
                                                patience=10,
                                                mu_init=0.01, mu_min=0.001, mu_max=0.5,
                                                schedule_mode='cosine', divergence_w=0.4, performance_w=0.3,
                                                lambda_ewc=0.4, ewc_sample_ratio=0.49, fisher_batch_size=32, ewc_new_client_epochs=2, fisher_refresh_round=15,
                                                lambda_min=0.2, lambda_max=0.8,
                                                trial_interval=5, max_trials_per_user=9, num_trials=3,
                                                weighting_strategy='quality',
                                                ewc_cache=None,
                                                seed=None):

    print(f"  FEDPROX + PERSONALIZATION + EWC NEW CLIENTS")
    print(f"  Case: {case.upper()} | Seed: {seed}")
    print(f"  Phase 1: {num_rounds_initial} rounds (Group 1 only)")
    print(f"  Phase 2: {num_rounds_combined} rounds (Group 1 + Group 2)")
    print(f"  Total:   {num_rounds_initial + num_rounds_combined} rounds")
    print(f"  Local epochs: {local_epochs} | Batch: {batch_size}")
    print(f"  FedProx: μ_init={mu_init}, range=[{mu_min}, {mu_max}]")
    print(f"  EWC: λ_range=[{lambda_min}, {lambda_max}], "
          f"refresh@round {fisher_refresh_round}")
    
    print(f"\n  Case Configuration:")
    if case == 'case1':
        print(f"    G1 (Initial): Mostly LOW severity (single-session users)")
        print(f"    G2 (New):     HIGH/MOD/CRIT (multi-session users)")
        print(f"    Challenge:    Complex users adapt to simple baseline")
    elif case == 'case2':
        print(f"    G1 (Initial): HIGH/MOD/CRIT (multi-session users)")
        print(f"    G2 (New):     Mostly LOW severity (single-session users)")
        print(f"    Challenge:    Simple users benefit from complex baseline")
    elif case == 'case3':
        print(f"    G1 (Initial): Mixed severity (both HIGH/MOD/CRIT + LOW)")
        print(f"    G2 (New):     Mixed severity (both HIGH/MOD/CRIT + LOW)")
        print(f"    Challenge:    Heterogeneous adaptation in both groups")
    
    print(f"  Training Strategy:")
    print(f"    G1 → FedProx (μ-regularized)")
    print(f"       LOW: DASPR + FedProx on backbone (personalized)")
    print(f"       HIGH/MOD/CRIT: Simple CE + FedProx on all weights (single-head)")
    print(f"    G2 → EWC (λ-regularized)")
    print(f"       LOW: DASPR + EWC on shared layers (personalized)")
    print(f"       HIGH/MOD/CRIT: Simple CE + EWC on shared layers (single-head)")
    print(f"  FedAvg: backbone=quality-weighted, global_output=equal-weighted")

    monitor      = ResourceMonitor()
    total_rounds = num_rounds_initial + num_rounds_combined
    global_model = compile_global_model(input_shape, n_class)

    # Severity Detection
    client_severities       = {}
    client_severity_metrics = {}
    all_per_user            = {**group1_per_user, **group2_per_user}

    for uid in sorted(all_per_user.keys()):
        s = all_per_user[uid]
        val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
        severity, metrics = detect_severity_distribution_only(
            s['train_labels'], s[val_key_labels], n_class)
        client_severities[uid]       = severity
        client_severity_metrics[uid] = metrics
        group_label = "G1" if uid in group1_per_user else "G2"
        head_type   = "DUAL" if severity in PERSONAL_SEVERITIES else "SINGLE"
        print(f"    {uid:8s} [{group_label}]: {severity:8s} [{head_type:6s}] "
              f"(score={metrics['score']:2.0f}, "
              f"overlap={metrics['overlap_ratio']:.2f})")

    print(f"\n  Severity Distribution by Group:")
    for grp_label, grp_dict in [('G1', group1_per_user), ('G2', group2_per_user)]:
        sev_counts = {}
        for uid in grp_dict:
            sev = client_severities[uid]
            sev_counts[sev] = sev_counts.get(sev, 0) + 1
        print(f"    {grp_label}: {dict(sev_counts)}")

    # Create Group 1 models 
    print(f"\n  Creating Group 1 Client Models:")
    group1_models = {}
    for uid in sorted(group1_per_user.keys()):
        s = group1_per_user[uid]
        val_key_data   = 'val_data'   if 'val_data'   in s else 'test_data'
        val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
        group1_models[uid] = create_model_for_user(
            uid, input_shape, n_class, client_severities[uid],
            s['train_labels'], s[val_key_labels], s[val_key_data],
            global_model, len(s['train_data']), get_user_domain(uid),
            num_trials=num_trials)

    mu_controller_g1 = AdaptiveMuController(
        list(group1_per_user.keys()), mu_min, mu_max, mu_init,
        total_rounds, schedule_mode, divergence_w, performance_w, 0.3)

    # Track comprehensive metrics
    per_round_per_user_val_metrics  = {}  
    per_round_per_user_test_metrics = {}  
    per_round_global_acc            = []

    user_best_val_acc = {uid: 0.0   for uid in all_per_user}
    user_patience_ctr = {uid: 0     for uid in all_per_user}
    user_stopped      = {uid: False for uid in all_per_user}
    prev_val_acc      = {uid: None  for uid in all_per_user}
    trial_counts      = {uid: 0     for uid in all_per_user}

    fisher_numpy            = None
    layer_to_weights        = None
    reference_weights_numpy = None
    group2_models           = {}
    adaptive_lambdas        = {}
    mu_controller_g2        = None


    # PHASE 1: Initial Clients Only
    print(f"  PHASE 1: Initial Clients Only (Rounds 1–{num_rounds_initial})")
    print(f"  Strategy: FedProx for all G1 clients")

    for round_num in range(num_rounds_initial):
        monitor.mark_round_start(round_num)
        print(f"\n{'─'*65}\n  Round {round_num+1}/{total_rounds} "
              f"[PHASE 1 - {case.upper()}]\n{'─'*65}")

        global_backbone  = get_backbone_weights(global_model)
        global_output_ws = get_global_output_weights(global_model)

        if round_num > 0 and round_num % trial_interval == 0:
            for uid in sorted(group1_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user):
                    s = group1_per_user[uid]
                    val_key_data   = 'val_data'   if 'val_data'   in s else 'test_data'
                    val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'test_data':    s[val_key_data],
                        'test_labels':  s[val_key_labels],}
                    new_model, improved = run_lightweight_trial(
                        uid, group1_models[uid], trial_splits, global_model,
                        input_shape, n_class, get_user_domain(uid),
                        client_severities[uid], 3, 2, batch_size)
                    if improved:
                        del group1_models[uid]
                        group1_models[uid] = new_model
                    trial_counts[uid] += 1
                    gc.collect()

        # Local training 
        round_contributors = {}
        for uid in sorted(group1_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group1_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            try:
                mu_user = mu_controller_g1.update_mu(
                    round_num, uid,
                    [w for ws in global_backbone.values() for w in ws],
                    prev_val_acc[uid], verbose=False)

                set_backbone_weights(group1_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group1_models[uid], global_output_ws)

                history = train_local_fedprox_unified_v2(
                    group1_models[uid], global_backbone,
                    splits['train_data'], splits['train_labels'],
                    mu_user, local_epochs, batch_size,
                    splits[val_key_data], splits[val_key_labels],
                    client_severity=client_severities[uid],
                    round_num=round_num,
                    total_rounds=total_rounds)

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                prev_val_acc[uid]       = val_acc
                round_contributors[uid] = group1_models[uid]
                mu_controller_g1.save_local_weights(uid, group1_models[uid])

                sev  = client_severities[uid]
                head = "D" if sev in PERSONAL_SEVERITIES else "S"
                print(f"    {uid:8s}[{sev:4s}|{head}] "
                      f"tr={train_acc:.4f} val={val_acc:.4f} "
                      f"μ={mu_user:.5f}")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}: ❌ {e}")
                import traceback; traceback.print_exc()
            finally:
                gc.collect()

        # FedAvg 
        if round_contributors:
            quality_weights = compute_quality_weights(
                list(round_contributors.keys()),
                group1_per_user,
                client_severity_metrics)
            equal_weights = {uid: 1.0 / len(round_contributors)
                             for uid in round_contributors}

            if round_num % 10 == 0:
                print(f"\n    FedAvg Weights (Phase 1 - {case.upper()}):")
                for uid in sorted(round_contributors.keys()):
                    print(f"      {uid}: backbone={quality_weights[uid]:.4f}, "
                          f"global_out={equal_weights[uid]:.4f} "
                          f"({client_severities[uid]})")

            averaged_backbone = federated_avg_backbone(
                round_contributors, quality_weights,
                verbose=(round_num % 10 == 0))
            if averaged_backbone:
                set_backbone_weights(global_model, averaged_backbone)

            avg_global_output = federated_avg_global_output(
                round_contributors, equal_weights,
                verbose=(round_num % 10 == 0))
            if avg_global_output:
                set_global_output_weights(global_model, avg_global_output)

            if round_num % 10 == 0:
                print(f"    FedAvg complete: "
                      f"{len(round_contributors)} contributors")

        per_round_per_user_val_metrics[round_num]  = {}
        per_round_per_user_test_metrics[round_num] = {}

        for uid in group1_per_user:
            sev    = client_severities[uid]
            splits = group1_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            per_round_per_user_val_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group1_models[uid], splits[val_key_data],
                splits[val_key_labels], sev, batch_size,
                global_model=global_model)
            
            per_round_per_user_test_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group1_models[uid], splits['test_data'],
                splits['test_labels'], sev, batch_size,
                global_model=global_model)

        print(f"\n  Post-FedAvg Metrics [Phase 1 - {case.upper()}]:")
        print(f"  {'User':<10} {'Sev':<8} {'H':<2} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
        print(f"  {'─'*52}")
        for uid in sorted(group1_per_user.keys()):
            sev     = client_severities[uid]
            head    = "D" if sev in PERSONAL_SEVERITIES else "S"
            metrics = per_round_per_user_test_metrics[round_num][uid]
            print(f"  {uid:<10} {sev:<8} {head:<2} "
                  f"{metrics['accuracy']:>6.4f} "
                  f"{metrics['precision_weighted']:>6.4f} "
                  f"{metrics['recall_weighted']:>6.4f} "
                  f"{metrics['f1_weighted']:>6.4f}")

        global_acc = evaluate_with_generator(
            global_model, group1_splits['test_data'],
            group1_splits['test_labels'], batch_size)
        per_round_global_acc.append(global_acc)

        all_test_acc = [m['accuracy'] for m in per_round_per_user_test_metrics[round_num].values()]
        print(f"\n    G1 TEST: "
              f"{np.mean(all_test_acc):.4f}±{np.std(all_test_acc):.4f} | "
              f"Global pool: {global_acc:.4f}")
        print(f"    Stopped: "
              f"{sum(user_stopped[u] for u in group1_per_user)}/"
              f"{len(group1_per_user)}")

        if round_num % 5 == 0:
            mu_controller_g1.round_summary(round_num)

        monitor.mark_round_end(round_num)
        gc.collect()



    shared_layer_indices = []
    for idx, layer in enumerate(global_model.layers):
        if any(bb_name in layer.name for bb_name in BACKBONE_LAYER_NAMES):
            shared_layer_indices.append(idx)

    print(f"\n  Computing Federated Fisher Information from Group 1...")
    fisher_numpy, layer_to_weights = create_fisher_federated(
        model=global_model,
        per_user_splits=group1_per_user,
        shared_layer_indices=shared_layer_indices,
        batch_size=fisher_batch_size,
        sample_ratio=ewc_sample_ratio,
        aggregation='quality_weighted',
        severity_metrics=client_severity_metrics)

    reference_weights_numpy = global_model.get_weights()
    print(f"  Federated Fisher computed & reference weights stored")

    if ewc_cache is not None:
        ewc_cache.build(
            reference_weights_numpy=reference_weights_numpy,
            fisher_numpy=fisher_numpy,
            layer_to_weights=layer_to_weights,
            lambda_ewc=lambda_ewc)
        print(f"  EWCCache built: {len(ewc_cache._cache)} entries")

    gc.collect()

    print(f"\n  Creating Group 2 Client Models:")
    for uid in sorted(group2_per_user.keys()):
        s = group2_per_user[uid]
        val_key_data   = 'val_data'   if 'val_data'   in s else 'test_data'
        val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
        group2_models[uid] = create_model_for_user(
            uid, input_shape, n_class, client_severities[uid],
            s['train_labels'], s[val_key_labels], s[val_key_data],
            global_model, len(s['train_data']), get_user_domain(uid),
            num_trials=num_trials)

    mu_controller_g2 = AdaptiveMuController(
        list(group2_per_user.keys()), mu_min, mu_max, mu_init,
        total_rounds, schedule_mode, divergence_w, performance_w, 0.3)

    print(f"\n  Computing adaptive λ for Group 2 clients "
          f"[range: {lambda_min}–{lambda_max}]")

    for uid, splits in group2_per_user.items():
        current_acc = evaluate_user_model(
            group2_models[uid],
            splits.get('val_data',   splits['test_data']),
            splits.get('val_labels', splits['test_labels']),
            client_severities[uid], batch_size, global_model=global_model)

        n_samples    = len(splits['train_data'])
        size_factor  = float(np.clip(1.0 - (n_samples / 2000.0), 0.0, 1.0))

        class_counts     = np.sum(splits['train_labels'], axis=0)
        active_classes   = int(np.sum(class_counts > 0))
        n_total_classes  = splits['train_labels'].shape[1]
        diversity_factor = 1.0 - (active_classes / n_total_classes)

        raw_lambda = (lambda_ewc
                      + 0.2 * (1.0 - current_acc)
                      + 0.2 * size_factor
                      + 0.1 * diversity_factor)
        adaptive_lambdas[uid] = float(
            np.clip(round(raw_lambda, 3), lambda_min, lambda_max))
        print(f"    {uid}: λ={adaptive_lambdas[uid]:.3f} "
              f"[acc={current_acc:.3f} | n={n_samples} | "
              f"classes={active_classes}/{n_total_classes}]")

    lam_vals = list(adaptive_lambdas.values())
    print(f"  λ range: {min(lam_vals):.3f} – {max(lam_vals):.3f} "
          f"(avg={np.mean(lam_vals):.3f})")

    print(f"\n  EWC Warm-up for Group 2 New Clients:")
    for uid, splits in group2_per_user.items():
        try:
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            ewc_history = train_new_client_with_ewc(
                model=group2_models[uid],
                train_data=splits['train_data'],
                train_labels=splits['train_labels'],
                reference_weights_numpy=reference_weights_numpy,
                fisher_numpy=fisher_numpy,
                layer_to_weights=layer_to_weights,
                lambda_ewc=adaptive_lambdas[uid],
                epochs=ewc_new_client_epochs,
                batch_size=batch_size,
                val_data=splits[val_key_data],
                val_labels=splits[val_key_labels],
                patience=patience,
                verbose=0,
                ewc_cache=None)

            final_val = (ewc_history['val_accuracy'][-1]
                         if ewc_history['val_accuracy'] else 0.0)
            print(f"    {uid}: warm-up done | val={final_val:.4f} | "
                  f"λ={adaptive_lambdas[uid]:.3f}")

        except Exception as e:
            print(f"    {uid}: EWC warm-up failed — {e}")
        finally:
            gc.collect()


    # PHASE 2: All Clients Together
    print(f"  PHASE 2: All Clients ({case.upper()})")
    print(f"  Rounds {num_rounds_initial+1}–{total_rounds}")
    print(f"  Fisher refresh scheduled at Round {fisher_refresh_round}")
    print(f"  G1 → FedProx | G2 → EWC")

    combined_test_data   = np.concatenate(
        [group1_splits['test_data'],   group2_splits['test_data']],   axis=0)
    combined_test_labels = np.concatenate(
        [group1_splits['test_labels'], group2_splits['test_labels']], axis=0)

    for round_num in range(num_rounds_initial, total_rounds):
        monitor.mark_round_start(round_num)
        current_round = round_num + 1
        print(f"\n{'─'*65}\n  Round {current_round}/{total_rounds} "
              f"[PHASE 2 - {case.upper()}]\n{'─'*65}")

        global_backbone  = get_backbone_weights(global_model)
        global_output_ws = get_global_output_weights(global_model)

        if round_num > 0 and round_num % trial_interval == 0:
            for uid in sorted(group1_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user):
                    s = group1_per_user[uid]
                    val_key_data   = 'val_data'   if 'val_data'   in s else 'test_data'
                    val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'test_data':    s[val_key_data],
                        'test_labels':  s[val_key_labels],
                    }
                    new_model, improved = run_lightweight_trial(
                        uid, group1_models[uid], trial_splits, global_model,
                        input_shape, n_class, get_user_domain(uid),
                        client_severities[uid], 3, 2, batch_size)
                    if improved:
                        del group1_models[uid]
                        group1_models[uid] = new_model
                    trial_counts[uid] += 1
                    gc.collect()

            for uid in sorted(group2_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user):
                    s = group2_per_user[uid]
                    val_key_data   = 'val_data'   if 'val_data'   in s else 'test_data'
                    val_key_labels = 'val_labels' if 'val_labels' in s else 'test_labels'
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'test_data':    s[val_key_data],
                        'test_labels':  s[val_key_labels],
                    }
                    new_model, improved = run_lightweight_trial(
                        uid, group2_models[uid], trial_splits, global_model,
                        input_shape, n_class, get_user_domain(uid),
                        client_severities[uid], 3, 2, batch_size)
                    if improved:
                        del group2_models[uid]
                        group2_models[uid] = new_model
                    trial_counts[uid] += 1
                    gc.collect()

        round_contributors = {}

        for uid in sorted(group1_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group1_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            try:
                mu_user = mu_controller_g1.update_mu(
                    round_num, uid,
                    [w for ws in global_backbone.values() for w in ws],
                    prev_val_acc[uid], verbose=False)

                set_backbone_weights(group1_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group1_models[uid], global_output_ws)

                history = train_local_fedprox_unified_v2(
                    group1_models[uid], global_backbone,
                    splits['train_data'], splits['train_labels'],
                    mu_user, local_epochs, batch_size,
                    splits[val_key_data], splits[val_key_labels],
                    client_severity=client_severities[uid],
                    round_num=round_num,
                    total_rounds=total_rounds)

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                prev_val_acc[uid]       = val_acc
                round_contributors[uid] = group1_models[uid]
                mu_controller_g1.save_local_weights(uid, group1_models[uid])

                sev  = client_severities[uid]
                head = "D" if sev in PERSONAL_SEVERITIES else "S"
                print(f"    {uid:8s}[{sev:4s}|{head}] "
                      f"tr={train_acc:.4f} val={val_acc:.4f} "
                      f"μ={mu_user:.5f}")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}:  {e}")
                import traceback; traceback.print_exc()
            finally:
                gc.collect()

        for uid in sorted(group2_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group2_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            try:
                mu_user = mu_controller_g2.update_mu(
                    round_num, uid,
                    [w for ws in global_backbone.values() for w in ws],
                    prev_val_acc[uid], verbose=False)

                set_backbone_weights(group2_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group2_models[uid], global_output_ws)

                history = train_local_fedprox_with_ewc(
                    local_model=group2_models[uid],
                    global_backbone_weights=global_backbone,
                    train_data=splits['train_data'],
                    train_labels=splits['train_labels'],
                    mu=0.0,
                    epochs=local_epochs,
                    batch_size=batch_size,
                    val_data=splits[val_key_data],
                    val_labels=splits[val_key_labels],
                    client_severity=client_severities[uid],
                    reference_weights_numpy=reference_weights_numpy,
                    fisher_numpy=fisher_numpy,
                    layer_to_weights=layer_to_weights,
                    lambda_ewc=adaptive_lambdas[uid],
                    round_num=round_num,
                    total_rounds=total_rounds,
                    ewc_cache=None)

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                ewc_loss  = history.get('ewc_loss', [0.0])[-1]
                prev_val_acc[uid]       = val_acc
                round_contributors[uid] = group2_models[uid]
                mu_controller_g2.save_local_weights(uid, group2_models[uid])

                sev  = client_severities[uid]
                head = "D" if sev in PERSONAL_SEVERITIES else "S"
                print(f"    {uid:8s}[{sev:4s}|{head}] "
                      f"tr={train_acc:.4f} val={val_acc:.4f} "
                      f"ewc={ewc_loss:.6f} [λ={adaptive_lambdas[uid]:.3f}]")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}:  {e}")
                import traceback; traceback.print_exc()
            finally:
                gc.collect()

        # FedAvg (unchanged)
        if round_contributors:
            all_contributor_splits = {}
            for uid in round_contributors:
                if uid in group1_per_user:
                    all_contributor_splits[uid] = group1_per_user[uid]
                else:
                    all_contributor_splits[uid] = group2_per_user[uid]

            quality_weights = compute_quality_weights(
                list(round_contributors.keys()),
                all_contributor_splits,
                client_severity_metrics)
            equal_weights = {uid: 1.0 / len(round_contributors)
                             for uid in round_contributors}

            if round_num % 10 == 0:
                print(f"\n    FedAvg Weights (Phase 2 - {case.upper()}):")
                print(f"    Backbone: quality-weighted | "
                      f"Global_output: equal-weighted")
                for uid in sorted(round_contributors.keys()):
                    grp = "G1" if uid in group1_per_user else "G2"
                    print(f"      {uid} [{grp}]: "
                          f"backbone={quality_weights[uid]:.4f}, "
                          f"global_out={equal_weights[uid]:.4f} "
                          f"({client_severities[uid]})")

            averaged_backbone = federated_avg_backbone(
                round_contributors, quality_weights,
                verbose=(round_num % 10 == 0))
            if averaged_backbone:
                set_backbone_weights(global_model, averaged_backbone)

            avg_global_output = federated_avg_global_output(
                round_contributors, equal_weights,
                verbose=(round_num % 10 == 0))
            if avg_global_output:
                set_global_output_weights(global_model, avg_global_output)

            if round_num % 10 == 0:
                print(f"  FedAvg complete: "
                      f"{len(round_contributors)} contributors "
                      f"(G1={sum(1 for u in round_contributors if u in group1_per_user)}, "
                      f"G2={sum(1 for u in round_contributors if u in group2_per_user)})")

        # Fisher refresh (unchanged)
        if current_round % fisher_refresh_round == 0:
            print(f"  Federated Fisher Refresh at Round {current_round}")

            fisher_numpy = refresh_fisher_federated(
                model=global_model,
                per_user_splits=group1_per_user,
                shared_layer_indices=shared_layer_indices,
                old_fisher_numpy=fisher_numpy,
                layer_to_weights=layer_to_weights,
                batch_size=fisher_batch_size,
                sample_ratio=ewc_sample_ratio,
                ema_alpha=0.3,
                aggregation='quality_weighted',
                severity_metrics=client_severity_metrics)

            reference_weights_numpy = global_model.get_weights()
            print(f"  Federated Fisher refreshed & reference weights updated")

            if ewc_cache is not None:
                ewc_cache.refresh(
                    reference_weights_numpy=reference_weights_numpy,
                    fisher_numpy=fisher_numpy,
                    lambda_ewc=lambda_ewc)
                print(f"  EWCCache refreshed (base λ={lambda_ewc})")

            print(f"\n  Updating adaptive λ after Fisher refresh...")
            for uid, splits in group2_per_user.items():
                val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
                val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'
                current_acc = evaluate_user_model(
                    group2_models[uid],
                    splits[val_key_data], splits[val_key_labels],
                    client_severities[uid], batch_size,
                    global_model=global_model)

                n_samples        = len(splits['train_data'])
                size_factor      = float(
                    np.clip(1.0 - (n_samples / 2000.0), 0.0, 1.0))
                class_counts     = np.sum(splits['train_labels'], axis=0)
                active_classes   = int(np.sum(class_counts > 0))
                n_total_classes  = splits['train_labels'].shape[1]
                diversity_factor = 1.0 - (active_classes / n_total_classes)

                raw_lambda = (lambda_ewc
                              + 0.2 * (1.0 - current_acc)
                              + 0.2 * size_factor
                              + 0.1 * diversity_factor)
                adaptive_lambdas[uid] = float(
                    np.clip(round(raw_lambda, 3), lambda_min, lambda_max))
                print(f"    {uid}: λ={adaptive_lambdas[uid]:.3f} "
                      f"[acc={current_acc:.3f}]")

            lam_vals = list(adaptive_lambdas.values())
            print(f"  λ range after refresh: "
                  f"{min(lam_vals):.3f}–{max(lam_vals):.3f}")

            gc.collect()

        per_round_per_user_val_metrics[round_num]  = {}
        per_round_per_user_test_metrics[round_num] = {}

        for uid in group1_per_user:
            sev    = client_severities[uid]
            splits = group1_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            per_round_per_user_val_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group1_models[uid], splits[val_key_data],
                splits[val_key_labels], sev, batch_size,
                global_model=global_model)
            
            per_round_per_user_test_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group1_models[uid], splits['test_data'],
                splits['test_labels'], sev, batch_size,
                global_model=global_model)

        for uid in group2_per_user:
            sev    = client_severities[uid]
            splits = group2_per_user[uid]
            val_key_data   = 'val_data'   if 'val_data'   in splits else 'test_data'
            val_key_labels = 'val_labels' if 'val_labels' in splits else 'test_labels'

            per_round_per_user_val_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group2_models[uid], splits[val_key_data],
                splits[val_key_labels], sev, batch_size,
                global_model=global_model)
            
            per_round_per_user_test_metrics[round_num][uid] = evaluate_user_model_comprehensive(
                group2_models[uid], splits['test_data'],
                splits['test_labels'], sev, batch_size,
                global_model=global_model)

        print(f"\n  Post-FedAvg Metrics [Phase 2 - {case.upper()}]:")
        print(f"  {'User':<10} {'Grp':>4} {'Sev':>5} {'H':>2} "
              f"{'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'λ':>7}")
        for uid in sorted(all_per_user.keys()):
            sev       = client_severities[uid]
            head      = "D" if sev in PERSONAL_SEVERITIES else "S"
            grp       = "G1" if uid in group1_per_user else "G2"
            metrics   = per_round_per_user_test_metrics[round_num][uid]
            lam_str   = (f"{adaptive_lambdas.get(uid, 0):.3f}"
                         if grp == "G2" else "  —  ")
            print(f"  {uid:<10} {grp:>4} {sev:>5} {head:>2} "
                  f"{metrics['accuracy']:>6.4f} "
                  f"{metrics['precision_weighted']:>6.4f} "
                  f"{metrics['recall_weighted']:>6.4f} "
                  f"{metrics['f1_weighted']:>6.4f} {lam_str:>7}")

        global_acc = evaluate_with_generator(
            global_model, combined_test_data,
            combined_test_labels, batch_size)
        per_round_global_acc.append(global_acc)

        g1_test  = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                    for u in group1_per_user]
        g2_test  = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                    for u in group2_per_user]
        all_test = g1_test + g2_test

        low_test  = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                     for u in all_per_user
                     if client_severities[u] in PERSONAL_SEVERITIES]
        high_test = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                     for u in all_per_user
                     if client_severities[u] in GLOBAL_SEVERITIES]

        print(f"\n    G1 TEST:  {np.mean(g1_test):.4f}±{np.std(g1_test):.4f}")
        print(f"    G2 TEST:  {np.mean(g2_test):.4f}±{np.std(g2_test):.4f}")
        print(f"    ALL TEST: "
              f"{np.mean(all_test):.4f}±{np.std(all_test):.4f} | "
              f"Global: {global_acc:.4f}")
        if low_test:
            print(f" LOW ({len(low_test)}): "
                  f"test={np.mean(low_test):.4f}±{np.std(low_test):.4f}")
        if high_test:
            print(f"  MOD/HIGH/CRIT ({len(high_test)}): "
                  f"test={np.mean(high_test):.4f}±{np.std(high_test):.4f}")
        print(f"    G1-G2 Gap: "
              f"{np.mean(g1_test) - np.mean(g2_test):+.4f}")
        print(f"    Stopped: "
              f"{sum(user_stopped.values())}/{len(all_per_user)}")

        if round_num % 5 == 0:
            mu_controller_g1.round_summary(round_num)
            if mu_controller_g2 is not None:
                mu_controller_g2.round_summary(round_num)

        monitor.mark_round_end(round_num)
        gc.collect()

    # FINAL SUMMARY WITH ALL METRICS
    print(f"\n{'═'*70}")
    print(f"  TRAINING COMPLETE ({case.upper()} - Seed: {seed})")
    print(f"{'═'*70}")
    print(f"  Phase 1: {num_rounds_initial} rounds (Group 1 only)")
    print(f"  Phase 2: {num_rounds_combined} rounds (All clients)")
    print(f"  Total:   {total_rounds} rounds")
    print(f"  Fisher refresh: Round {fisher_refresh_round} [FEDERATED + EMA]")

    last_round = total_rounds - 1
    final_test_metrics = per_round_per_user_test_metrics[last_round]

    print(f"\n  Final Results by User:")
    print(f"  {'User':<10} {'Grp':>4} {'Sev':>9} {'Head':>6} "
          f"{'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'λ':>7} {'T':>3}")
    print(f"  {'─'*70}")
    for uid in sorted(all_per_user.keys()):
        sev     = client_severities[uid]
        head    = "DUAL" if sev in PERSONAL_SEVERITIES else "SINGLE"
        grp     = "G1" if uid in group1_per_user else "G2"
        t       = trial_counts.get(uid, 0)
        metrics = final_test_metrics[uid]
        lam_str = (f"{adaptive_lambdas.get(uid, 0):.3f}"
                   if grp == "G2" else "  —  ")
        print(f"  {uid:<10} {grp:>4} {sev:>9} {head:>6} "
              f"{metrics['accuracy']:>6.4f} "
              f"{metrics['precision_weighted']:>6.4f} "
              f"{metrics['recall_weighted']:>6.4f} "
              f"{metrics['f1_weighted']:>6.4f} {lam_str:>7} {t:>3}")

    # Aggregate by group with all metrics
    g1_metrics = [final_test_metrics[u] for u in group1_per_user]
    g2_metrics = [final_test_metrics[u] for u in group2_per_user]

    print(f"\n  By Group (TEST):")
    print(f"    {'Group':<4} {'N':>3} {'Acc':>7} {'Precision':>10} {'Recall':>10} {'F1':>10}")

    for grp_label, grp_metrics in [('G1', g1_metrics), ('G2', g2_metrics)]:
        acc_vals  = [m['accuracy'] for m in grp_metrics]
        prec_vals = [m['precision_weighted'] for m in grp_metrics]
        rec_vals  = [m['recall_weighted'] for m in grp_metrics]
        f1_vals   = [m['f1_weighted'] for m in grp_metrics]
        
        print(f"    {grp_label:<4} {len(grp_metrics):>3} "
              f"{np.mean(acc_vals):>7.4f} "
              f"{np.mean(prec_vals):>10.4f} "
              f"{np.mean(rec_vals):>10.4f} "
              f"{np.mean(f1_vals):>10.4f}")

    # Aggregate by severity with all metrics
    print(f"\n  By Severity (TEST):")
    print(f"    {'Severity':<10} {'N':>3} {'Acc':>7} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"    {'─'*55}")
    for sev_level in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
        users = [uid for uid, s in client_severities.items() if s == sev_level]
        if users:
            metrics_list = [final_test_metrics[uid] for uid in users]
            acc_vals  = [m['accuracy'] for m in metrics_list]
            prec_vals = [m['precision_weighted'] for m in metrics_list]
            rec_vals  = [m['recall_weighted'] for m in metrics_list]
            f1_vals   = [m['f1_weighted'] for m in metrics_list]
            
            print(f"    {sev_level:<10} {len(users):>3} "
                  f"{np.mean(acc_vals):>7.4f} "
                  f"{np.mean(prec_vals):>10.4f} "
                  f"{np.mean(rec_vals):>10.4f} "
                  f"{np.mean(f1_vals):>10.4f}")

    # Case-specific insights 
    print(f"\n  Case-Specific Insights ({case.upper()}):")
    if case == 'case1':
        print(f"    - G1 (LOW severity) established simple baseline")
        print(f"    - G2 (HIGH/MOD) complex users adapted via EWC")
        print(f"    - Challenge: Complex patterns learning from simple baseline")
    elif case == 'case2':
        print(f"    - G1 (HIGH/MOD) established complex baseline")
        print(f"    - G2 (LOW severity) simple users joined")
        print(f"    - Challenge: Simple users benefiting from complex knowledge")
    elif case == 'case3':
        print(f"    - G1 (Mixed) heterogeneous initial group")
        print(f"    - G2 (Mixed) heterogeneous new group")
        print(f"    - Challenge: Balanced adaptation across severities")

    # Overall summary
    all_test_metrics = list(final_test_metrics.values())
    print(f"\n  Overall TEST:")
    print(f"    Accuracy:  {np.mean([m['accuracy'] for m in all_test_metrics]):.4f}±{np.std([m['accuracy'] for m in all_test_metrics]):.4f}")
    print(f"    Precision: {np.mean([m['precision_weighted'] for m in all_test_metrics]):.4f}±{np.std([m['precision_weighted'] for m in all_test_metrics]):.4f}")
    print(f"    Recall:    {np.mean([m['recall_weighted'] for m in all_test_metrics]):.4f}±{np.std([m['recall_weighted'] for m in all_test_metrics]):.4f}")
    print(f"    F1-Score:  {np.mean([m['f1_weighted'] for m in all_test_metrics]):.4f}±{np.std([m['f1_weighted'] for m in all_test_metrics]):.4f}")
    print(f"  Global pool:  {per_round_global_acc[-1]:.4f}")
    print(f"  Total trials: {sum(trial_counts.values())}")
    print(f"  Early stopped: "
          f"{sum(user_stopped.values())}/{len(all_per_user)}")

    g1_final_test_acc = [final_test_metrics[u]['accuracy'] for u in group1_per_user]
    g2_final_test_acc = [final_test_metrics[u]['accuracy'] for u in group2_per_user]
    
    if np.mean(g2_final_test_acc) >= np.mean(g1_final_test_acc) - 0.05:
        print(f"\n  New clients successfully adapted "
              f"(within 5% of initial clients)")
    else:
        print(f"\n  New clients still catching up — "
              f"consider more Phase 2 rounds")

    monitor.print_summary()

    return {
        'global_model'                  : global_model,
        'group1_models'                 : group1_models,
        'group2_models'                 : group2_models,
        'client_severities'             : client_severities,
        'client_severity_metrics'       : client_severity_metrics,
        'group1_user_ids'               : list(group1_per_user.keys()),
        'group2_user_ids'               : list(group2_per_user.keys()),
        'per_round_per_user_val_metrics': per_round_per_user_val_metrics,
        'per_round_per_user_test_metrics': per_round_per_user_test_metrics,
        'per_round_per_user_val_acc'    : {r: {u: m['accuracy'] for u, m in metrics.items()} 
                                            for r, metrics in per_round_per_user_val_metrics.items()},
        'per_round_per_user_test_acc'   : {r: {u: m['accuracy'] for u, m in metrics.items()} 
                                            for r, metrics in per_round_per_user_test_metrics.items()},
        
        'per_round_global_acc'          : per_round_global_acc,
        'mu_controller_g1'              : mu_controller_g1,
        'mu_controller_g2'              : mu_controller_g2,
        'trial_counts'                  : trial_counts,
        'adaptive_lambdas'              : adaptive_lambdas,
        'fisher_numpy'                  : fisher_numpy,
        'layer_to_weights'              : layer_to_weights,
        'final_global_acc'              : per_round_global_acc[-1],
        'final_metrics_summary'         : {
            'all': {
                'accuracy': np.mean([m['accuracy'] for m in all_test_metrics]),
                'precision': np.mean([m['precision_weighted'] for m in all_test_metrics]),
                'recall': np.mean([m['recall_weighted'] for m in all_test_metrics]),
                'f1': np.mean([m['f1_weighted'] for m in all_test_metrics]),
            },
            'g1': {
                'accuracy': np.mean([m['accuracy'] for m in g1_metrics]),
                'precision': np.mean([m['precision_weighted'] for m in g1_metrics]),
                'recall': np.mean([m['recall_weighted'] for m in g1_metrics]),
                'f1': np.mean([m['f1_weighted'] for m in g1_metrics]),
            },
            'g2': {
                'accuracy': np.mean([m['accuracy'] for m in g2_metrics]),
                'precision': np.mean([m['precision_weighted'] for m in g2_metrics]),
                'recall': np.mean([m['recall_weighted'] for m in g2_metrics]),
                'f1': np.mean([m['f1_weighted'] for m in g2_metrics]),
            }
        },
        'case': case,
    }

In [ ]:
# Prepare framework for multi-seed
def train_fedprox_personalized_with_new_clients_multi_seed(all_data_splits,
                                                           seeds, input_shape, n_class,
                                                           case='case1',
                                                           num_rounds_initial=5, num_rounds_combined=15,
                                                           local_epochs=2,
                                                           batch_size=32,
                                                           patience=10,
                                                           mu_init=0.01, mu_min=0.001, mu_max=0.5,
                                                           schedule_mode='cosine', divergence_w=0.4, performance_w=0.3,
                                                           lambda_ewc=0.4, ewc_sample_ratio=0.49, fisher_batch_size=32, ewc_new_client_epochs=2, fisher_refresh_round=20,
                                                           lambda_min=0.2, lambda_max=0.8,
                                                           trial_interval=5, max_trials_per_user=9, num_trials=3,
                                                           weighting_strategy='quality'):

    print(f"  MULTI-SEED TRAINING: {case.upper()}")
    print(f"  Seeds: {seeds}")

    all_results = {}
    ewc_cache = EWCCache()

    for seed_idx, seed in enumerate(seeds, 1):
        print(f"  SEED {seed_idx}/{len(seeds)}: {seed} ({case.upper()})")

        np.random.seed(seed)
        tf.random.set_seed(seed)
        random.seed(seed)

        if isinstance(all_data_splits, LazySeedDataLoader):
            seed_data = all_data_splits.load(seed)
        else:
            seed_data = all_data_splits[seed]

        try:
            results = train_fedprox_personalized_with_new_clients(
                group1_per_user      = seed_data['group1_per_user'],
                group1_splits        = seed_data['group1_splits'],
                group2_per_user      = seed_data['group2_per_user'],
                group2_splits        = seed_data['group2_splits'],
                input_shape          = input_shape,
                n_class              = n_class,
                case                 = case,
                num_rounds_initial   = num_rounds_initial,
                num_rounds_combined  = num_rounds_combined,
                local_epochs         = local_epochs,
                batch_size           = batch_size,
                patience             = patience,
                mu_init              = mu_init,
                mu_min               = mu_min,
                mu_max               = mu_max,
                schedule_mode        = schedule_mode,
                divergence_w         = divergence_w,
                performance_w        = performance_w,
                lambda_ewc           = lambda_ewc,
                ewc_sample_ratio     = ewc_sample_ratio,
                fisher_batch_size    = fisher_batch_size,
                ewc_new_client_epochs= ewc_new_client_epochs,
                fisher_refresh_round = fisher_refresh_round,
                lambda_min           = lambda_min,
                lambda_max           = lambda_max,
                ewc_cache            = ewc_cache,
                trial_interval       = trial_interval,
                max_trials_per_user  = max_trials_per_user,
                num_trials           = num_trials,
                weighting_strategy   = weighting_strategy,
                seed                 = seed)

            all_results[seed] = results

        except Exception as e:
            print(f"   SEED {seed} FAILED: {e}")
            import traceback; traceback.print_exc()
            all_results[seed] = {'error': str(e)}

        finally:
            if seed in all_results and 'error' not in all_results.get(seed, {}):
                r = all_results[seed]
                for key in ['global_model', 'group1_models', 'group2_models',
                            'mu_controller_g1', 'mu_controller_g2',
                            'fisher_numpy']:
                    if key in r:
                        obj = r.pop(key)
                        if key in ('group1_models', 'group2_models'):
                            for m in obj.values():
                                safe_model_delete(m)
                        else:
                            del obj

            ewc_cache._cache.clear()
            gc.collect()
            K.clear_session()
            time.sleep(2)

    print(f"  CROSS-SEED SUMMARY: {case.upper()}")
    valid_seeds = [s for s in seeds
                   if 'error' not in all_results.get(s, {'error': ''})]

    if valid_seeds:
        seed_metrics = {
            'accuracy': [], 'precision': [], 'recall': [], 'f1': [],
            'g1_accuracy': [], 'g1_precision': [], 'g1_recall': [], 'g1_f1': [],
            'g2_accuracy': [], 'g2_precision': [], 'g2_recall': [], 'g2_f1': [],
            'global_acc': []}
        
        for seed in valid_seeds:
            r = all_results[seed]
            summary = r['final_metrics_summary']
            
            # Overall metrics
            seed_metrics['accuracy'].append(summary['all']['accuracy'])
            seed_metrics['precision'].append(summary['all']['precision'])
            seed_metrics['recall'].append(summary['all']['recall'])
            seed_metrics['f1'].append(summary['all']['f1'])
            
            # G1 metrics
            seed_metrics['g1_accuracy'].append(summary['g1']['accuracy'])
            seed_metrics['g1_precision'].append(summary['g1']['precision'])
            seed_metrics['g1_recall'].append(summary['g1']['recall'])
            seed_metrics['g1_f1'].append(summary['g1']['f1'])
            
            # G2 metrics
            seed_metrics['g2_accuracy'].append(summary['g2']['accuracy'])
            seed_metrics['g2_precision'].append(summary['g2']['precision'])
            seed_metrics['g2_recall'].append(summary['g2']['recall'])
            seed_metrics['g2_f1'].append(summary['g2']['f1'])
            
            # Global
            seed_metrics['global_acc'].append(r['final_global_acc'])
            
            print(f"  Seed {seed:3d}: "
                  f"acc={summary['all']['accuracy']:.4f} "
                  f"prec={summary['all']['precision']:.4f} "
                  f"rec={summary['all']['recall']:.4f} "
                  f"f1={summary['all']['f1']:.4f} | "
                  f"trials={sum(r['trial_counts'].values())}")


        print(f"  Across {len(valid_seeds)} seeds ({case.upper()}):")
        print(f"  {'Metric':<15} {'Overall':>15} {'G1':>15} {'G2':>15}")
        
        for metric in ['accuracy', 'precision', 'recall', 'f1']:
            overall = f"{np.mean(seed_metrics[metric]):.4f}±{np.std(seed_metrics[metric]):.4f}"
            g1 = f"{np.mean(seed_metrics[f'g1_{metric}']):.4f}±{np.std(seed_metrics[f'g1_{metric}']):.4f}"
            g2 = f"{np.mean(seed_metrics[f'g2_{metric}']):.4f}±{np.std(seed_metrics[f'g2_{metric}']):.4f}"
            print(f"  {metric.capitalize():<15} {overall:>15} {g1:>15} {g2:>15}")
        
        print(f"  Global pool:    {np.mean(seed_metrics['global_acc']):.4f}±{np.std(seed_metrics['global_acc']):.4f}")
        
        # G1-G2 Gap
        g1_g2_gaps = [seed_metrics['g1_accuracy'][i] - seed_metrics['g2_accuracy'][i] 
                      for i in range(len(valid_seeds))]
        print(f"  G1-G2 Gap:      {np.mean(g1_g2_gaps):+.4f}±{np.std(g1_g2_gaps):.4f}")
        print(f"\n  Per-Severity (TEST) Across Seeds:")
        print(f"    {'Severity':<10} {'N':>4} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
        
        for sev_level in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
            all_sev_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
            
            for seed in valid_seeds:
                r           = all_results[seed]
                final_round = max(r['per_round_per_user_test_metrics'].keys())
                sev_users   = [uid for uid, s in r['client_severities'].items()
                               if s == sev_level]
                
                if sev_users:
                    for uid in sev_users:
                        m = r['per_round_per_user_test_metrics'][final_round][uid]
                        all_sev_metrics['accuracy'].append(m['accuracy'])
                        all_sev_metrics['precision'].append(m['precision_weighted'])
                        all_sev_metrics['recall'].append(m['recall_weighted'])
                        all_sev_metrics['f1'].append(m['f1_weighted'])
            
            if all_sev_metrics['accuracy']:
                head = "DUAL" if sev_level in PERSONAL_SEVERITIES else "SINGLE"
                print(f"    {sev_level:<10} {len(all_sev_metrics['accuracy']):>4} "
                      f"{np.mean(all_sev_metrics['accuracy']):>7.4f} "
                      f"{np.mean(all_sev_metrics['precision']):>7.4f} "
                      f"{np.mean(all_sev_metrics['recall']):>7.4f} "
                      f"{np.mean(all_sev_metrics['f1']):>7.4f} [{head}]")

        print(f"\n  Per-Group × Severity (TEST) Across Seeds:")
        print(f"    {'Group-Severity':<18} {'N':>4} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
        
        for grp_label in ['G1', 'G2']:
            for sev_level in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
                all_sev_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
                
                for seed in valid_seeds:
                    r           = all_results[seed]
                    final_round = max(r['per_round_per_user_test_metrics'].keys())

                    if grp_label == 'G1':
                        grp_users = set(r['group1_user_ids'])
                    else:
                        grp_users = set(r['group2_user_ids'])

                    sev_users = [uid for uid, s in r['client_severities'].items()
                                 if s == sev_level and uid in grp_users]
                    
                    if sev_users:
                        for uid in sev_users:
                            m = r['per_round_per_user_test_metrics'][final_round][uid]
                            all_sev_metrics['accuracy'].append(m['accuracy'])
                            all_sev_metrics['precision'].append(m['precision_weighted'])
                            all_sev_metrics['recall'].append(m['recall_weighted'])
                            all_sev_metrics['f1'].append(m['f1_weighted'])
                
                if all_sev_metrics['accuracy']:
                    head = "DUAL" if sev_level in PERSONAL_SEVERITIES else "SINGLE"
                    label = f"{grp_label} {sev_level}"
                    print(f"    {label:<18} {len(all_sev_metrics['accuracy']):>4} "
                          f"{np.mean(all_sev_metrics['accuracy']):>7.4f} "
                          f"{np.mean(all_sev_metrics['precision']):>7.4f} "
                          f"{np.mean(all_sev_metrics['recall']):>7.4f} "
                          f"{np.mean(all_sev_metrics['f1']):>7.4f} [{head}]")

        # Adaptation success rate
        adaptation_success = sum(
            1 for s in valid_seeds
            if seed_metrics['g2_accuracy'][valid_seeds.index(s)] >=
               seed_metrics['g1_accuracy'][valid_seeds.index(s)] - 0.05)
        print(f"\n  Adaptation Success (G2 within 5% of G1): "
              f"{adaptation_success}/{len(valid_seeds)} seeds "
              f"({100*adaptation_success/len(valid_seeds):.1f}%)")

        print(f"\n  Case-Specific Interpretation ({case.upper()}):")
        if case == 'case1':
            print(f"    - Initial group (G1): Mostly LOW severity users")
            print(f"    - New group (G2): HIGH/MOD/CRIT users")
            print(f"    - Expected: G2 may struggle initially (complex patterns)")
            print(f"    - Success: G2 adapts to simple baseline via EWC")
            if np.mean(seed_metrics['g2_accuracy']) >= np.mean(seed_metrics['g1_accuracy']) - 0.05:
                print(f"     Complex users successfully adapted!")
                print(f"       Precision: {np.mean(seed_metrics['g2_precision']):.4f}")
                print(f"       Recall:    {np.mean(seed_metrics['g2_recall']):.4f}")
                print(f"       F1-Score:  {np.mean(seed_metrics['g2_f1']):.4f}")
            else:
                print(f"   Complex users need more adaptation rounds")
                print(f"       Current F1: {np.mean(seed_metrics['g2_f1']):.4f}")
        
        elif case == 'case2':
            print(f"    - Initial group (G1): HIGH/MOD/CRIT users")
            print(f"    - New group (G2): Mostly LOW severity users")
            print(f"    - Expected: G2 should benefit from rich baseline")
            print(f"    - Success: G2 leverages complex knowledge via EWC")
            if np.mean(seed_metrics['g2_accuracy']) >= np.mean(seed_metrics['g1_accuracy']):
                print(f"  Simple users matched or exceeded complex baseline!")
                print(f"       G2 F1-Score: {np.mean(seed_metrics['g2_f1']):.4f}")
                print(f"       G1 F1-Score: {np.mean(seed_metrics['g1_f1']):.4f}")
            else:
                print(f"     Simple users learning from complex baseline")
                print(f"       G2 Precision: {np.mean(seed_metrics['g2_precision']):.4f}")
                print(f"       G2 Recall:    {np.mean(seed_metrics['g2_recall']):.4f}")
        
        elif case == 'case3':
            print(f"    - Initial group (G1): Mixed severity (heterogeneous)")
            print(f"    - New group (G2): Mixed severity (heterogeneous)")
            print(f"    - Expected: Balanced adaptation across all severities")
            print(f"    - Success: Both groups benefit from diverse knowledge")
            if np.mean(seed_metrics['g2_accuracy']) >= np.mean(seed_metrics['g1_accuracy']) - 0.05:
                print(f"    Heterogeneous groups adapted successfully!")
                print(f"       Overall F1: {np.mean(seed_metrics['f1']):.4f}±{np.std(seed_metrics['f1']):.4f}")
            else:
                print(f"    Some severity levels need more rounds")
                print(f"       G1-G2 F1 Gap: {np.mean(seed_metrics['g1_f1']) - np.mean(seed_metrics['g2_f1']):+.4f}")

        # NEW: Performance distribution analysis
        print(f"\n  Performance Distribution Analysis:")
        print(f"    Metric ranges across {len(valid_seeds)} seeds:")
        for metric in ['accuracy', 'precision', 'recall', 'f1']:
            vals = seed_metrics[metric]
            print(f"      {metric.capitalize():<12}: "
                  f"[{np.min(vals):.4f}, {np.max(vals):.4f}] "
                  f"(median={np.median(vals):.4f})")

    else:
        print("  No successful seeds!")

    failed_seeds = [s for s in seeds if s not in valid_seeds]
    if failed_seeds:
        print(f"\n  Failed seeds: {failed_seeds}")

    return all_results

In [ ]:
SEEDS = [42, 100]
input_shape = (38, 20, 20, 1)
n_class = 12

# Run the full pipeline
loader_case1 = LazySeedDataLoader(fl_clients, case='case1')

results_case1 = train_fedprox_personalized_with_new_clients_multi_seed(
    all_data_splits=loader_case1,
    seeds=SEEDS,
    input_shape=input_shape,
    n_class=n_class,
    case='case1',  
    num_rounds_initial=10,
    num_rounds_combined=40,
    local_epochs=2,
    batch_size=32,
    patience=10,
    mu_init=0.01,
    mu_min=0.001,
    mu_max=0.5,
    lambda_ewc=0.4,
    lambda_min=0.2,
    lambda_max=0.8,
    fisher_refresh_round=15,
    trial_interval=5,
    max_trials_per_user=9)

del loader_case1
gc.collect()

In [ ]:
SEEDS = [42, 100]
input_shape = (38, 20, 20, 1)
n_class = 12

# Run the full pipeline
loader_case2 = LazySeedDataLoader(fl_clients, case='case2')

results_case2 = train_fedprox_personalized_with_new_clients_multi_seed(
    all_data_splits=loader_case2,
    seeds=SEEDS,
    input_shape=input_shape,
    n_class=n_class,
    case='case2', 
    num_rounds_initial=10,
    num_rounds_combined=40,
    local_epochs=2,
    batch_size=32,
    patience=10,
    mu_init=0.01,
    mu_min=0.001,
    mu_max=0.5,
    lambda_ewc=0.4,
    lambda_min=0.2,
    lambda_max=0.8,
    fisher_refresh_round=15,
    trial_interval=5,
    max_trials_per_user=9)

del loader_case2
gc.collect()

In [ ]:
SEEDS = [42, 100]
input_shape = (38, 20, 20, 1)
n_class = 12

# Run the full pipeline
loader_case3 = LazySeedDataLoader(fl_clients, case='case3')

results_case3 = train_fedprox_personalized_with_new_clients_multi_seed(
    all_data_splits=loader_case3,
    seeds=SEEDS,
    input_shape=input_shape,
    n_class=n_class,
    case='case3', 
    num_rounds_initial=10,
    num_rounds_combined=40,
    local_epochs=2,
    batch_size=32,
    patience=10,
    mu_init=0.01,
    mu_min=0.001,
    mu_max=0.5,
    lambda_ewc=0.4,
    lambda_min=0.2,
    lambda_max=0.8,
    fisher_refresh_round=15,
    trial_interval=5,
    max_trials_per_user=9)


del loader_case3
gc.collect()